# Tarea 2: Detección de organismos marinos con Deep Learning

**Dataset:** [FjordFish v1.1](https://zenodo.org/records/17950781) (Poling et al., 2025, CC-BY-4.0). Son 3002 imágenes de cámaras submarinas remotas (RUV) en las costas de Noruega y Suecia, con 6350 cajas anotadas en formato YOLO sobre 28 clases de peces y crustáceos.

El notebook corre en Google Colab (GPU T4) o en local. Todo lo costoso se guarda en Drive y se reutiliza: el dataset preprocesado, los checkpoints y las predicciones. Si una sesión se corta, basta con volver a ejecutar todo; cada entrenamiento continúa desde su último checkpoint.

- **`CONFIG["fast_dev_run"] = True`** corre todo con un subconjunto chico y 1-2 épocas por modelo, para validar el notebook en minutos.
- **`CONFIG["run"]`** activa cada entrenamiento por separado. Así los modelos se pueden repartir entre varias cuentas de Colab que comparten la carpeta `Tarea2_Deteccion` de Drive (cada integrante agrega un acceso directo en "Mi unidad").
- **Una vez entrenado todo**, una corrida con todos los switches apagados regenera las tablas y figuras desde las predicciones guardadas, sin usar la GPU.

## Estructura del notebook
| Parte | Contenido |
|---|---|
| Setup | instalación, rutas, `CONFIG` |
| 0 | descarga, preprocesamiento, EDA, taxonomía y partición train/val/test |
| Componentes comunes | dataset, transformaciones, evaluador único y medición de eficiencia |
| 1 | modelos desde cero: CNN propia (estilo CenterNet) y YOLO11s |
| 2 | transfer learning desde COCO: YOLO11s y Faster R-CNN |
| 3 | robustez del mejor modelo frente a degradaciones del conjunto de prueba |
| Cierre | comparación final y costo computacional |

## Integrantes
Alex Marambio, Diego Pérez, Aarón Pozas

## Setup

In [ ]:
import os
import subprocess
import sys
from importlib import metadata

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Versiones fijas: Ultralytics cambia hiperparámetros por defecto entre releases y eso movería los resultados.
PINNED = {"ultralytics": "8.4.177", "torchmetrics": "1.9.0", "faster-coco-eval": "1.8.0"}
for dist, version in PINNED.items():
    try:
        installed = metadata.version(dist)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{dist}=={version}"], check=True)
print("IN_COLAB:", IN_COLAB)

In [ ]:
import copy
import hashlib
import json
import math
import random
import shutil
import time
import urllib.request
import zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import ultralytics
import yaml
from matplotlib import patches
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from PIL import Image, ImageOps
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from torch.utils.flop_counter import FlopCounterMode
from torchmetrics.detection import MeanAveragePrecision
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, decode_image, read_file
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d, box_iou
from torchvision.transforms import v2 as T
from torchvision.transforms.v2 import functional as TF
from ultralytics import YOLO

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Tarea2_Deteccion")
    # Las imágenes se leen desde el disco de la VM; leerlas desde Drive en cada época sería mucho más lento.
    WORK_ROOT = Path("/content/work")
else:
    DRIVE_ROOT = Path.cwd() / "local_run" / "Tarea2_Deteccion"
    WORK_ROOT = DRIVE_ROOT / "work"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("WORK_ROOT:", WORK_ROOT)

In [ ]:
CONFIG = {
    "seed": 42,

    # True = subconjunto chico y 1-2 épocas por modelo, para validar el notebook de punta a punta.
    # Poner en False para la corrida real en Colab.
    "fast_dev_run": False,
    "fast_dev_sizes": {"train": 32, "val": 16, "test": 16},

    # Entrenamientos a ejecutar en esta sesión. Lo que ya está entrenado se reutiliza desde Drive
    # aunque su switch esté apagado.
    "run": {"cnn": True, "yolo_scratch": True, "yolo_coco": True, "frcnn": True, "robustness": True},
    # Los checkpoints se escriben en el disco de la VM y se respaldan en Drive cada tantos minutos y al terminar.
    "checkpoint_sync_minutes": 15,

    "data": {
        "zenodo_url": "https://zenodo.org/api/records/17950781/files/FjordFish.zip/content",
        "zip_md5": "951bf139c3e412fa7924fa7ffc34a307",
        "work_size": 640,              # lado largo de trabajo, común a los 4 modelos
        "jpeg_quality": 95,
        "apply_exif_transpose": False,  # ver la verificación EXIF de la sección 0.1
        "split_fracs": [0.70, 0.15, 0.15],
        "near_dup_hamming": 10,        # dHash a <= 10 bits de distancia = casi duplicado
        "scene_window": 3,             # vecinos por id con los que se compara cada imagen
        "max_group_frac": 0.08,        # una escena más grande que esto se parte en bloques
        "group_block": 30,
        "split_trials": 2000,
    },

    "eval": {
        "conf_min": 0.001,   # las métricas tipo COCO necesitan la cola de baja confianza
        "max_det": 100,
        "nms_iou": 0.7,      # solo YOLO; Faster R-CNN usa su NMS por clase de 0.5
        "iou_match": 0.5,    # IoU para precisión/recall/F1 y matriz de confusión
        "min_test_instances": 10,
        "latency_images": 100,
    },

    "cnn": {
        "input_hw": [384, 640],            # alto x ancho del letterbox, múltiplos de 32 cercanos a 16:9
        "widths": [64, 128, 256, 384],     # canales de las etapas con stride 4, 8, 16, 32
        "depths": [1, 2, 2, 1],            # bloques residuales por etapa
        "neck_channels": 128,
        "head_channels": 64,
        "max_objects": 128,
        "batch_size": 16,
        "epochs": 120,
        "lr": 1e-3,
        "weight_decay": 0.05,
        "warmup_epochs": 3,
        "min_lr_ratio": 0.01,
        "ema_decay": 0.9998,
        "grad_clip": 10.0,
        "eval_every": 5,
        "patience": 30,                    # épocas sin mejorar val mAP50-95 antes de detenerse
        "size_loss_weight": 0.1,
        "offset_loss_weight": 1.0,
        "crop_min_scale": 0.6,
        "crop_prob": 0.5,
        "lr_test_iters": 300,
        "overfit_iters": 150,
    },

    # Valores compartidos por los tres entrenamientos YOLO; cada etapa sobrescribe lo suyo.
    "yolo_common": {
        "imgsz": 640, "batch": 16, "cache": "ram", "deterministic": True, "amp": True,
        "optimizer": "SGD", "momentum": 0.937, "weight_decay": 5e-4, "lrf": 0.01,
        "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4, "fliplr": 0.5, "flipud": 0.0,
        "scale": 0.5, "translate": 0.1, "mosaic": 1.0, "mixup": 0.0,
    },
    "yolo_scratch": {
        "model": "yolo11s.yaml", "pretrained": False,
        "epochs": 200, "patience": 50, "lr0": 0.01, "warmup_epochs": 3, "close_mosaic": 15,
    },
    "yolo_coco": {
        "model": "yolo11s.pt",
        "stage_a": {"epochs": 15, "patience": 15, "freeze": 11, "lr0": 0.01, "warmup_epochs": 2, "close_mosaic": 5},
        "stage_b": {"epochs": 50, "patience": 20, "lr0": 0.002, "warmup_epochs": 1, "warmup_bias_lr": 0.002,
                    "close_mosaic": 10},
    },

    "frcnn": {
        "batch_size": 4,
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "warmup_iters": 500,
        "min_lr_ratio": 0.01,
        "stage_a": {"epochs": 5, "lr": 0.005},    # lr = 0.02 * batch / 16 (regla lineal de torchvision)
        "stage_b": {"epochs": 12, "lr": 0.001},
    },

    "robustness": {
        "levels": {
            "low_light": [0.5, 0.3, 0.15],   # factor de brillo
            "blur": [1.0, 2.5, 5.0],         # sigma del desenfoque gaussiano (px a 640)
            "noise": [10, 25, 50],           # sigma del ruido gaussiano (escala 0-255)
            "contrast": [0.6, 0.4, 0.2],     # factor de contraste respecto de la media
            "turbidity": [0.7, 0.5, 0.3],    # transmisión t del velo de turbidez
        },
    },
}

if CONFIG["fast_dev_run"]:
    CONFIG["cnn"].update(epochs=2, eval_every=1, patience=2, batch_size=8, lr_test_iters=20, overfit_iters=20)
    CONFIG["yolo_common"].update(imgsz=320, batch=8)
    for stage in (CONFIG["yolo_scratch"], CONFIG["yolo_coco"]["stage_a"], CONFIG["yolo_coco"]["stage_b"]):
        stage.update(epochs=1, patience=1, close_mosaic=0)
    CONFIG["frcnn"]["stage_a"]["epochs"] = 1
    CONFIG["frcnn"]["stage_b"]["epochs"] = 1
    CONFIG["frcnn"]["warmup_iters"] = 4
    CONFIG["eval"]["latency_images"] = 5


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
# En Windows los DataLoader con workers usan spawn y no pueden serializar las clases definidas en el notebook.
NUM_WORKERS = 0 if os.name == "nt" else 2
CONFIG["yolo_common"].update(seed=CONFIG["seed"], workers=NUM_WORKERS)

ENV_INFO = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "ultralytics": ultralytics.__version__,
    "torchmetrics": metadata.version("torchmetrics"),
    "device": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
}
print(json.dumps(ENV_INFO, indent=2))

In [ ]:
SCALE_TAG = "fast_dev" if CONFIG["fast_dev_run"] else "full"
SPLITS_DIR = DRIVE_ROOT / "splits"
PROCESSED_ZIP = DRIVE_ROOT / "data" / "processed.zip"
RAW_DIR = WORK_ROOT / "raw"
PROCESSED_DIR = WORK_ROOT / "processed"
YOLO_DATA_DIR = WORK_ROOT / "yolo_data"
RUNS_DIR = DRIVE_ROOT / "runs" / SCALE_TAG        # respaldo de checkpoints, historiales y predicciones
CKPT_ROOT = WORK_ROOT / "checkpoints" / SCALE_TAG  # checkpoints en uso (disco de la VM)
PRED_DIR = RUNS_DIR / "predictions"
RESULTS_DIR = DRIVE_ROOT / "results" / SCALE_TAG  # figuras y métricas del informe
for d in (SPLITS_DIR, PROCESSED_ZIP.parent, RAW_DIR, RUNS_DIR, PRED_DIR / "robustness", RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)


def results_dir(part):
    d = RESULTS_DIR / part
    d.mkdir(parents=True, exist_ok=True)
    return d


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))


def read_json(path):
    return json.loads(Path(path).read_text())


def atomic_torch_save(obj, path):
    """Escribe a un temporal y renombra: un corte de sesión a mitad de escritura no deja un checkpoint corrupto."""
    tmp = Path(f"{path}.tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def save_fig(fig, path):
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()


# Paleta categórica en orden fijo: cada modelo y cada degradación conserva su color en todas las figuras.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ_CMAP = LinearSegmentedColormap.from_list(
    "seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281", "#0d366b"])
DIV_CMAP = LinearSegmentedColormap.from_list("div_blue_red", ["#256abf", "#f0efec", "#d03b3b"])
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "axes.titlesize": 11, "axes.prop_cycle": plt.cycler(color=PALETTE), "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linestyle": "-", "grid.linewidth": 0.6,
    "xtick.color": "#898781", "ytick.color": "#898781", "text.color": "#0b0b0b",
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 9,
})

## Parte 0: Datos

### 0.1 Descarga, taxonomía y preprocesamiento

**Por qué preprocesar una sola vez**: el zip original pesa 6 GB y mezcla tres tipos de imagen:
- frames de video de 1920×1080;
- frames 4K guardados como PNG con extensión `.jpg`;
- fotos de cámara de hasta 4288×2848.

Los cuatro modelos trabajan con lado largo 640, así que todo se redimensiona una vez (lado largo 640, JPEG q95) y se guarda en Drive un zip de unos 300 MB; las sesiones siguientes solo lo copian al disco de la VM. Además, Colab tiene 2 vCPU: si hubiera que decodificar imágenes 4K en cada época, la GPU pasaría la mayor parte del tiempo esperando. PIL abre cada archivo según su contenido y no según su extensión, así que los PNG disfrazados se leen bien.

**Por qué consolidar clases**: el desbalance es extremo. goldsinny tiene 1433 cajas, mientras que lobster y black_goby tienen 1 cada una, y sharks_unknown no tiene ninguna. Con un test del 15 %, una especie de menos de 30 cajas queda con 0 a 4 instancias en test. Su AP sería ruido y, en los modelos entrenados desde cero, prácticamente imposible de aprender. Por eso esas especies se fusionan en la clase genérica de su familia, que el propio dataset ya usa para individuos no identificables, y se eliminan las clases vacías:

| Original | Clase de entrenamiento |
|---|---|
| haddock (7), poor_cod (22) | gadids_unknown |
| hermit (15), lobster (1) | decapods_unknown |
| two_spot_goby (113), black_goby (1), sand_goby (3) | gobies |
| sharks_unknown (0), unknown (0) | se eliminan |

Las demás especies se conservan. ling (30) y cuckoo_male (31) quedan justo en el límite y se mantienen como clase propia.

In [ ]:
ORIGINAL_CLASSES = [
    "ballan", "cuckoo_female", "cuckoo_male", "corkwing_female", "corkwing_male", "goldsinny", "rock_cook",
    "labrids_unknown", "cod", "cod_yoy", "haddock", "pollack", "poor_cod", "saithe", "whiting", "gadids_unknown",
    "spiny_dogfish", "sharks_unknown", "brown_crab", "green_crab", "hermit", "lobster", "decapods_unknown", "ling",
    "two_spot_goby", "black_goby", "sand_goby", "unknown",
]
CLASS_MERGE = {
    "haddock": "gadids_unknown", "poor_cod": "gadids_unknown",
    "hermit": "decapods_unknown", "lobster": "decapods_unknown",
    "two_spot_goby": "gobies", "black_goby": "gobies", "sand_goby": "gobies",
    "sharks_unknown": None, "unknown": None,
}
_target_names = [CLASS_MERGE.get(c, c) for c in ORIGINAL_CLASSES]
CLASSES = list(dict.fromkeys(name for name in _target_names if name is not None))
NUM_CLASSES = len(CLASSES)
ORIG_TO_NEW = {i: (CLASSES.index(name) if name is not None else None) for i, name in enumerate(_target_names)}
print(f"{len(ORIGINAL_CLASSES)} clases originales -> {NUM_CLASSES} clases de entrenamiento")
print(CLASSES)

In [ ]:
def md5_of(path, chunk_size=1 << 24):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_raw_zip():
    zip_path = RAW_DIR / "FjordFish.zip"
    if zip_path.exists() and md5_of(zip_path) == CONFIG["data"]["zip_md5"]:
        return zip_path
    tmp = zip_path.with_suffix(".part")
    print("Descargando FjordFish.zip (~6 GB) desde Zenodo...")
    urllib.request.urlretrieve(CONFIG["data"]["zenodo_url"], tmp)
    if md5_of(tmp) != CONFIG["data"]["zip_md5"]:
        raise RuntimeError("El md5 del zip descargado no coincide con el publicado en Zenodo")
    os.replace(tmp, zip_path)
    return zip_path


def extract_raw(zip_path):
    if not any(RAW_DIR.rglob("data.yaml")):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(RAW_DIR)
    return next(RAW_DIR.rglob("data.yaml")).parent


def list_raw_images(root):
    rows = []
    for split in ("train", "val"):
        for path in sorted((root / "images" / split).iterdir()):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                rows.append({"id": int(path.stem), "orig_split": split, "image": path,
                             "label": root / "labels" / split / f"{path.stem}.txt"})
    df = pd.DataFrame(rows)
    assert df["id"].is_unique, "Hay ids de imagen repetidos entre train y val"
    return df.sort_values("id").reset_index(drop=True)


def read_yolo_labels(path):
    """Filas (clase, xc, yc, w, h) normalizadas; un archivo ausente equivale a una imagen sin objetos."""
    path = Path(path)
    if not path.exists():
        return np.zeros((0, 5), np.float32)
    rows = [line.split()[:5] for line in path.read_text().splitlines() if line.strip()]
    return np.array(rows, dtype=np.float32).reshape(-1, 5)


def exif_orientation(path):
    with Image.open(path) as im:
        return im.getexif().get(0x0112, 1)


def draw_boxes(ax, boxes, color="white", captions=None, linestyle="-"):
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color,
                                       linewidth=1.5, linestyle=linestyle))
        if captions is not None:
            ax.text(x1, y1 - 3, captions[i], color="white", fontsize=7,
                    bbox=dict(facecolor=color, alpha=0.75, pad=1, edgecolor="none"))


def yolo_to_xyxy(labels, w, h):
    xc, yc, bw, bh = labels[:, 1], labels[:, 2], labels[:, 3], labels[:, 4]
    return np.stack([(xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h], 1)

In [ ]:
NEED_RAW = not (PROCESSED_DIR / "manifest.csv").exists() and not PROCESSED_ZIP.exists()
if NEED_RAW:
    RAW_ROOT = extract_raw(download_raw_zip())
    RAW_DF = list_raw_images(RAW_ROOT)
    print(RAW_DF["orig_split"].value_counts().to_dict())
else:
    print("Dataset preprocesado disponible; no se descarga el zip original.")

**Verificación de orientación EXIF.** Algunas fotos de cámara pueden traer un tag de orientación. Según la librería, la imagen se lee rotada o no, y si las cajas se anotaron sobre la otra orientación quedan desplazadas. Antes de preprocesar se dibujan las cajas sobre ambas versiones, y la que calza define `CONFIG["data"]["apply_exif_transpose"]`.

In [ ]:
if NEED_RAW:
    RAW_DF["exif_orientation"] = [exif_orientation(p) for p in RAW_DF["image"]]
    rotated = RAW_DF[RAW_DF["exif_orientation"] != 1]
    print("Imágenes con orientación EXIF distinta de 1:", len(rotated))
    if len(rotated):
        sample = rotated.head(4)
        fig, axes = plt.subplots(2, len(sample), figsize=(4 * len(sample), 7), squeeze=False)
        for col, row in enumerate(sample.itertuples()):
            labels = read_yolo_labels(row.label)
            names = [ORIGINAL_CLASSES[int(c)] for c in labels[:, 0]]
            with Image.open(row.image) as im:
                variants = {"píxeles tal cual": im.convert("RGB"),
                            "con exif_transpose": ImageOps.exif_transpose(im).convert("RGB")}
            for r, (title, img) in enumerate(variants.items()):
                ax = axes[r, col]
                ax.imshow(img)
                draw_boxes(ax, yolo_to_xyxy(labels, *img.size), PALETTE[0], names)
                ax.set_title(f"id {row.id}: {title}")
                ax.axis("off")
        save_fig(fig, results_dir("parte0") / "exif_check.png")

In [ ]:
def dhash64(image):
    """Hash perceptual de 64 bits: compara la luminancia de píxeles vecinos en una miniatura de 9x8."""
    small = np.asarray(image.convert("L").resize((9, 8), Image.Resampling.BILINEAR), dtype=np.int16)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    return int.from_bytes(np.packbits(bits).tobytes(), "big")


def preprocess_one(row):
    size = CONFIG["data"]["work_size"]
    with Image.open(row.image) as im:
        fmt, (orig_w, orig_h) = im.format, im.size
        scale = min(1.0, size / max(orig_w, orig_h))
        # draft decodifica los JPEG directamente a escala reducida: mucho más rápido en las imágenes grandes
        im.draft("RGB", (round(orig_w * scale), round(orig_h * scale)))
        if CONFIG["data"]["apply_exif_transpose"]:
            im = ImageOps.exif_transpose(im)
        im = im.convert("RGB")
    scale = size / max(im.size)
    if scale < 1:
        im = im.resize((round(im.width * scale), round(im.height * scale)), Image.Resampling.LANCZOS)
    im.save(PROCESSED_DIR / "images" / f"{row.id}.jpg", "JPEG", quality=CONFIG["data"]["jpeg_quality"])

    raw = read_yolo_labels(row.label)
    box_rows, lines = [], []
    for cls, xc, yc, bw, bh in raw:
        new_cls = ORIG_TO_NEW.get(int(cls))
        x1, y1 = max(0.0, xc - bw / 2), max(0.0, yc - bh / 2)
        x2, y2 = min(1.0, xc + bw / 2), min(1.0, yc + bh / 2)
        valid = new_cls is not None and x2 > x1 and y2 > y1
        box_rows.append({"id": row.id, "orig_cls": int(cls), "cls": new_cls if valid else -1,
                         "xc": (x1 + x2) / 2, "yc": (y1 + y2) / 2, "w": x2 - x1, "h": y2 - y1})
        if valid:
            lines.append(f"{new_cls} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {x2 - x1:.6f} {y2 - y1:.6f}")
    (PROCESSED_DIR / "labels" / f"{row.id}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))

    record = {"id": row.id, "orig_split": row.orig_split, "orig_w": orig_w, "orig_h": orig_h,
              "w": im.width, "h": im.height, "format": fmt, "exif_orientation": row.exif_orientation,
              "dhash": f"{dhash64(im):016x}", "n_boxes": len(lines)}
    return record, box_rows


if (PROCESSED_DIR / "manifest.csv").exists():
    print("Usando", PROCESSED_DIR)
elif PROCESSED_ZIP.exists():
    print("Restaurando el dataset preprocesado desde Drive...")
    with zipfile.ZipFile(PROCESSED_ZIP) as zf:
        zf.extractall(WORK_ROOT)
else:
    t0 = time.time()
    (PROCESSED_DIR / "images").mkdir(parents=True, exist_ok=True)
    (PROCESSED_DIR / "labels").mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        results = list(pool.map(preprocess_one, RAW_DF.itertuples()))
    pd.DataFrame([r for r, _ in results]).sort_values("id").to_csv(PROCESSED_DIR / "manifest.csv", index=False)
    pd.DataFrame([b for _, boxes in results for b in boxes]).to_csv(PROCESSED_DIR / "boxes.csv", index=False)
    local_zip = shutil.make_archive(str(WORK_ROOT / "processed"), "zip", root_dir=WORK_ROOT, base_dir="processed")
    shutil.copy(local_zip, PROCESSED_ZIP)
    print(f"Preprocesamiento listo en {time.time() - t0:.0f} s; copia en {PROCESSED_ZIP}")

MANIFEST = pd.read_csv(PROCESSED_DIR / "manifest.csv", dtype={"dhash": str}).sort_values("id").reset_index(drop=True)
BOXES = pd.read_csv(PROCESSED_DIR / "boxes.csv")
MANIFEST_BY_ID = MANIFEST.set_index("id")
print(f"{len(MANIFEST)} imágenes, {len(BOXES)} cajas originales, {(BOXES['cls'] >= 0).sum()} cajas de entrenamiento")

### 0.2 Análisis exploratorio

In [ ]:
summary_data = {
    "imagenes": len(MANIFEST),
    "cajas_originales": len(BOXES),
    "cajas_entrenamiento": int((BOXES["cls"] >= 0).sum()),
    "cajas_eliminadas": int((BOXES["cls"] < 0).sum()),
    "imagenes_sin_cajas": int((MANIFEST["n_boxes"] == 0).sum()),
    "max_cajas_por_imagen": int(MANIFEST["n_boxes"].max()),
    "imagenes_png_con_extension_jpg": int((MANIFEST["format"] == "PNG").sum()),
    "imagenes_con_orientacion_exif": int((MANIFEST["exif_orientation"] != 1).sum()),
}
print(json.dumps(summary_data, indent=2))
resolutions = (MANIFEST.groupby(["orig_w", "orig_h", "format"]).size().rename("imagenes")
               .reset_index().sort_values("imagenes", ascending=False))
display(resolutions)

In [ ]:
orig_counts = BOXES["orig_cls"].value_counts().reindex(range(len(ORIGINAL_CLASSES)), fill_value=0)
new_counts = BOXES.loc[BOXES["cls"] >= 0, "cls"].value_counts().reindex(range(NUM_CLASSES), fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
order = np.argsort(orig_counts.to_numpy())
kept = np.array([CLASS_MERGE.get(c, c) == c for c in ORIGINAL_CLASSES])
axes[0].barh(np.array(ORIGINAL_CLASSES)[order], orig_counts.to_numpy()[order],
             color=np.where(kept[order], PALETTE[0], "#c3c2b7"))
axes[0].set_xscale("log")
axes[0].set_title("28 clases originales (gris: fusionadas o eliminadas)")
axes[0].set_xlabel("cajas (escala log)")
order = np.argsort(new_counts.to_numpy())
axes[1].barh(np.array(CLASSES)[order], new_counts.to_numpy()[order], color=PALETTE[0])
axes[1].set_xscale("log")
axes[1].set_title(f"{NUM_CLASSES} clases de entrenamiento")
axes[1].set_xlabel("cajas (escala log)")
for ax in axes:
    ax.grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, results_dir("parte0") / "class_distribution.png")

In [ ]:
kept_boxes = BOXES[BOXES["cls"] >= 0].merge(MANIFEST[["id", "w", "h"]].rename(columns={"w": "img_w", "h": "img_h"}), on="id")
box_w_px = kept_boxes["w"] * kept_boxes["img_w"]
box_h_px = kept_boxes["h"] * kept_boxes["img_h"]
box_side = np.sqrt(box_w_px * box_h_px)
small_frac = float((box_side < 32).mean())
print(f"Cajas pequeñas (<32x32 px a 640): {small_frac:.1%}   medianas: {float(((box_side >= 32) & (box_side < 96)).mean()):.1%}"
      f"   grandes: {float((box_side >= 96).mean()):.1%}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(MANIFEST["n_boxes"], bins=np.arange(MANIFEST["n_boxes"].max() + 2) - 0.5, color=PALETTE[0])
axes[0].set_title("Cajas por imagen")
axes[0].set_xlabel("cajas")
axes[0].set_ylabel("imágenes")
axes[1].hist(box_side, bins=60, color=PALETTE[0])
for limit in (32, 96):
    axes[1].axvline(limit, color="#52514e", linewidth=1)
axes[1].set_title("Tamaño de caja, sqrt(área) a 640 px (líneas: límites COCO 32 y 96)")
axes[1].set_xlabel("px")
axes[2].scatter(box_w_px, box_h_px, s=4, alpha=0.3, color=PALETTE[0])
axes[2].set_title("Ancho vs alto de las cajas (px a 640)")
axes[2].set_xlabel("ancho")
axes[2].set_ylabel("alto")
fig.tight_layout()
save_fig(fig, results_dir("parte0") / "box_statistics.png")

In [ ]:
def load_gt(img_id):
    row = MANIFEST_BY_ID.loc[img_id]
    labels = read_yolo_labels(PROCESSED_DIR / "labels" / f"{img_id}.txt")
    return yolo_to_xyxy(labels, row["w"], row["h"]).astype(np.float32), labels[:, 0].astype(np.int64)


def image_path(img_id):
    return PROCESSED_DIR / "images" / f"{img_id}.jpg"


GT = {img_id: load_gt(img_id) for img_id in MANIFEST["id"]}

examples = MANIFEST[MANIFEST["n_boxes"] > 0].sample(6, random_state=CONFIG["seed"])["id"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, img_id in zip(axes.flat, examples):
    boxes, labels = GT[img_id]
    ax.imshow(Image.open(image_path(img_id)))
    draw_boxes(ax, boxes, PALETTE[0], [CLASSES[c] for c in labels])
    ax.set_title(f"id {img_id}")
    ax.axis("off")
fig.tight_layout()
save_fig(fig, results_dir("parte0") / "examples_gt.png")

### 0.3 Fuga entre particiones y partición propia

**Por qué una partición propia y agrupada**: FjordFish solo trae train y val (≈70/30) y no tiene test. Además, los ids consecutivos alternan entre train y val. Todo indica que la partición oficial se hizo imagen por imagen sobre frames de video, de cámaras fijas que graban la misma escena durante minutos. Si dos frames casi idénticos caen uno en train y otro en test, el modelo es evaluado sobre imágenes que en la práctica ya vio y el mAP sale optimista.

**Cómo se arma**:
1. Se calcula un hash perceptual (dHash, 64 bits) de cada imagen. Dos imágenes a ≤10 bits de distancia de Hamming se consideran casi duplicadas.
2. **Escenas**: se recorren los ids en orden y cada imagen se une con sus 3 vecinos siguientes si son casi duplicados. Esto solo encadena frames contiguos y no mezcla escenas distintas que por casualidad se parezcan. Una escena que supera el 8 % del dataset se parte en bloques de 30 frames.
3. **Asignación**: se prueban 2000 permutaciones aleatorias de escenas, cortadas en 70/15/15. Se elige la que minimiza el peor error de proporción por clase, con al menos una instancia de cada clase en val y en test.
4. La fuga residual se mide igual en ambas particiones: el porcentaje de imágenes de evaluación que tienen un casi duplicado en train.

Si `splits/split.csv` ya existe en Drive se reutiliza tal cual. Así los cuatro modelos, en cualquier cuenta y en cualquier sesión, usan exactamente las mismas particiones.

In [ ]:
_POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return _POPCOUNT8[x.view(np.uint8)].reshape(*x.shape, 8).sum(-1, dtype=np.int32)


def min_hamming(query, reference, chunk=256):
    """Distancia mínima de cada hash de query al conjunto reference."""
    out = [popcount64(np.bitwise_xor(query[i:i + chunk, None], reference[None, :])).min(1)
           for i in range(0, len(query), chunk)]
    return np.concatenate(out) if out else np.zeros(0, np.int32)


def scene_groups(hashes, threshold, window):
    parent = np.arange(len(hashes))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for k in range(1, window + 1):
        close = popcount64(np.bitwise_xor(hashes[:-k], hashes[k:])) <= threshold
        for i in np.flatnonzero(close):
            a, b = find(i), find(i + k)
            if a != b:
                parent[max(a, b)] = min(a, b)
    return pd.factorize(np.array([find(i) for i in range(len(hashes))]))[0]


def cap_groups(groups, max_size, block):
    out, next_id = groups.copy(), groups.max() + 1
    for g, size in Counter(groups).items():
        if size > max_size:
            members = np.flatnonzero(groups == g)
            for start in range(0, size, block):
                out[members[start:start + block]] = next_id
                next_id += 1
    return pd.factorize(out)[0]


def search_split(groups, class_matrix, fracs, trials, seed):
    n_groups = groups.max() + 1
    group_counts = np.zeros((n_groups, class_matrix.shape[1]))
    np.add.at(group_counts, groups, class_matrix)
    group_sizes = np.bincount(groups, minlength=n_groups)
    total_cls, total_imgs = group_counts.sum(0), group_sizes.sum()
    fracs = np.asarray(fracs)
    bounds = np.cumsum(fracs)[:-1]
    present = total_cls >= 3
    rng = np.random.default_rng(seed)
    best = {"score": np.inf}
    for _ in range(trials):
        order = rng.permutation(n_groups)
        # el centro de cada escena decide su partición, así ninguna escena queda repartida
        centers = (np.cumsum(group_sizes[order]) - group_sizes[order] / 2) / total_imgs
        split_of_group = np.empty(n_groups, dtype=int)
        split_of_group[order] = np.searchsorted(bounds, centers)
        per_split = np.stack([group_counts[split_of_group == s].sum(0) for s in range(3)])
        cls_err = np.abs(per_split[:, present] / total_cls[present] - fracs[:, None]).max()
        img_err = np.abs(np.bincount(split_of_group, weights=group_sizes, minlength=3) / total_imgs - fracs).max()
        missing = int((per_split[1:, present] == 0).sum())
        score = cls_err + img_err + missing
        if score < best["score"]:
            best = {"score": score, "split_of_group": split_of_group.copy(), "max_class_error": float(cls_err),
                    "max_image_error": float(img_err), "classes_missing_in_val_test": missing}
    return best


hashes = np.array([int(h, 16) for h in MANIFEST["dhash"]], dtype=np.uint64)
consecutive_dist = popcount64(np.bitwise_xor(hashes[:-1], hashes[1:]))
rng = np.random.default_rng(CONFIG["seed"])
random_pairs = rng.integers(0, len(hashes), size=(5000, 2))
random_dist = popcount64(np.bitwise_xor(hashes[random_pairs[:, 0]], hashes[random_pairs[:, 1]]))
NEAR_DUP = CONFIG["data"]["near_dup_hamming"]

split_csv = SPLITS_DIR / "split.csv"
if split_csv.exists():
    SPLIT_DF = pd.read_csv(split_csv)
    assert set(SPLIT_DF["id"]) == set(MANIFEST["id"]), "split.csv no corresponde a este dataset"
    SPLIT_DF = SPLIT_DF.set_index("id").loc[MANIFEST["id"]].reset_index()
    stats_path = SPLITS_DIR / "split_stats.json"
    search_info = read_json(stats_path)["search"] if stats_path.exists() else {}
    print("Partición existente reutilizada desde", split_csv)
else:
    groups = scene_groups(hashes, NEAR_DUP, CONFIG["data"]["scene_window"])
    groups = cap_groups(groups, int(CONFIG["data"]["max_group_frac"] * len(hashes)), CONFIG["data"]["group_block"])
    class_matrix = np.zeros((len(MANIFEST), NUM_CLASSES))
    row_of_id = {img_id: r for r, img_id in enumerate(MANIFEST["id"])}
    for img_id, cls in BOXES.loc[BOXES["cls"] >= 0, ["id", "cls"]].itertuples(index=False):
        class_matrix[row_of_id[img_id], cls] += 1
    search_info = search_split(groups, class_matrix, CONFIG["data"]["split_fracs"],
                               CONFIG["data"]["split_trials"], CONFIG["seed"])
    split_names = np.array(["train", "val", "test"])
    SPLIT_DF = pd.DataFrame({"id": MANIFEST["id"], "orig_split": MANIFEST["orig_split"], "group": groups,
                             "split": split_names[search_info.pop("split_of_group")[groups]]})
    SPLIT_DF.to_csv(split_csv, index=False)

SPLIT_IDS_FULL = {s: SPLIT_DF.loc[SPLIT_DF["split"] == s, "id"].tolist() for s in ("train", "val", "test")}

In [ ]:
def leakage(eval_mask, train_mask):
    dist = min_hamming(hashes[eval_mask], hashes[train_mask])
    return dist, float((dist <= NEAR_DUP).mean())


official_dist, official_leak = leakage((MANIFEST["orig_split"] == "val").to_numpy(),
                                       (MANIFEST["orig_split"] == "train").to_numpy())
own_dist, own_leak = leakage(SPLIT_DF["split"].isin(["val", "test"]).to_numpy(),
                             (SPLIT_DF["split"] == "train").to_numpy())

split_counts = {s: Counter(int(c) for i in ids for c in GT[i][1]) for s, ids in SPLIT_IDS_FULL.items()}
SPLIT_STATS = {
    "images": {s: len(ids) for s, ids in SPLIT_IDS_FULL.items()},
    "boxes": {s: sum(c.values()) for s, c in split_counts.items()},
    "boxes_per_class": {CLASSES[k]: {s: split_counts[s].get(k, 0) for s in SPLIT_IDS_FULL} for k in range(NUM_CLASSES)},
    "n_groups": int(SPLIT_DF["group"].nunique()),
    "largest_group": int(SPLIT_DF["group"].value_counts().max()),
    "near_dup_hamming": NEAR_DUP,
    "leakage_official_val_vs_train": official_leak,
    "leakage_own_valtest_vs_train": own_leak,
    "search": search_info,
}
write_json(SPLITS_DIR / "split_stats.json", SPLIT_STATS)
write_json(SPLITS_DIR / "class_map.json", {"classes": CLASSES, "original_classes": ORIGINAL_CLASSES,
                                           "merge": CLASS_MERGE})
for split, ids in SPLIT_IDS_FULL.items():
    (SPLITS_DIR / f"{split}.txt").write_text("\n".join(map(str, ids)) + "\n")

print(json.dumps({k: v for k, v in SPLIT_STATS.items() if k != "boxes_per_class"}, indent=2))
print(f"Fuga (casi duplicado en train): oficial {official_leak:.1%} -> propia {own_leak:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
bins = np.arange(66) - 0.5
axes[0].hist(random_dist, bins=bins, density=True, histtype="step", color=PALETTE[1], label="pares al azar")
axes[0].hist(consecutive_dist, bins=bins, density=True, histtype="step", color=PALETTE[0], label="ids consecutivos")
axes[0].axvline(NEAR_DUP, color="#52514e", linewidth=1)
axes[0].set_title("Distancia dHash: ids consecutivos vs pares al azar")
axes[0].set_xlabel("bits distintos (de 64)")
axes[0].legend()
axes[1].hist(official_dist, bins=bins, density=True, histtype="step", color=PALETTE[1],
             label=f"oficial (val vs train): {official_leak:.0%} casi duplicadas")
axes[1].hist(own_dist, bins=bins, density=True, histtype="step", color=PALETTE[0],
             label=f"propia (val+test vs train): {own_leak:.0%}")
axes[1].axvline(NEAR_DUP, color="#52514e", linewidth=1)
axes[1].set_title("Distancia de cada imagen de evaluación a la más parecida de train")
axes[1].set_xlabel("bits distintos (de 64)")
axes[1].legend()
y = np.arange(NUM_CLASSES)
total = np.array([sum(split_counts[s].get(k, 0) for s in SPLIT_IDS_FULL) for k in range(NUM_CLASSES)])
for offset, (split, color) in zip((-0.27, 0, 0.27), zip(SPLIT_IDS_FULL, PALETTE)):
    share = np.array([split_counts[split].get(k, 0) for k in range(NUM_CLASSES)]) / np.maximum(total, 1)
    axes[2].barh(y + offset, share, height=0.25, color=color, label=split)
for frac in sorted(set(CONFIG["data"]["split_fracs"])):
    axes[2].axvline(frac, color="#52514e", linewidth=1)
axes[2].set_yticks(y, CLASSES)
axes[2].set_title("Proporción de cajas de cada clase por partición")
axes[2].legend(loc="lower right")
axes[2].grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, results_dir("parte0") / "split_leakage.png")

In [ ]:
SPLIT_IDS = dict(SPLIT_IDS_FULL)
if CONFIG["fast_dev_run"]:
    sampler = random.Random(CONFIG["seed"])
    SPLIT_IDS = {s: sorted(sampler.sample(ids, min(len(ids), CONFIG["fast_dev_sizes"][s])))
                 for s, ids in SPLIT_IDS_FULL.items()}


def write_yolo_data(split_ids):
    """data.yaml con listas de rutas absolutas: Ultralytics resuelve las relativas contra el directorio actual."""
    YOLO_DATA_DIR.mkdir(parents=True, exist_ok=True)
    entries = {}
    for split, ids in split_ids.items():
        path = YOLO_DATA_DIR / f"{split}_{SCALE_TAG}.txt"
        path.write_text("\n".join(str(image_path(i).resolve()) for i in ids) + "\n")
        entries[split] = str(path)
    data_yaml = YOLO_DATA_DIR / f"data_{SCALE_TAG}.yaml"
    data_yaml.write_text(yaml.safe_dump({"path": str(PROCESSED_DIR.resolve()), **entries,
                                         "names": dict(enumerate(CLASSES))}, sort_keys=False))
    return data_yaml


DATA_YAML = write_yolo_data(SPLIT_IDS)
TRAIN_INSTANCES = Counter(int(c) for i in SPLIT_IDS_FULL["train"] for c in GT[i][1])
TEST_INSTANCES = Counter(int(c) for i in SPLIT_IDS["test"] for c in GT[i][1])
print({s: len(ids) for s, ids in SPLIT_IDS.items()}, "->", DATA_YAML)

## Componentes comunes

Lo que comparten los cuatro modelos:
- **Dataset en RAM**: imágenes uint8 a 640 decodificadas una sola vez. Las cajas se manejan como `tv_tensors.BoundingBoxes`, así las transformaciones de `torchvision.transforms.v2` mueven imagen y cajas juntas.
- **Evaluador único**: cada modelo guarda sus predicciones de val y test en el mismo formato (xyxy en píxeles de la imagen de 640, confianza ≥ 0.001, máximo 100 por imagen), y todas se miden con el mismo código. Así la comparación no depende de cómo cada librería calcula su mAP, y las tablas se pueden regenerar sin GPU.
- **Eficiencia**: parámetros, GFLOPs, latencia de punta a punta, memoria de GPU y tiempo de entrenamiento.
- **Checkpoints**: se escriben en el disco de la VM en cada época y se respaldan en Drive cada 15 minutos y al terminar. Un checkpoint pesa entre 50 y 350 MB, y escribirlo en Drive en cada época llenaría la cuota gratuita con versiones antiguas. Si la sesión se corta, se pierden como máximo 15 minutos de entrenamiento.

**Métricas**:
- **mAP50-95** (COCO, promedio de AP sobre IoU de 0.50 a 0.95) es la métrica principal, porque premia la localización precisa.
- **mAP50** es más indulgente con cajas algo desplazadas.
- **Precisión, recall y F1** se calculan a IoU 0.5 contando todas las cajas (micro-promedio). Usan un umbral de confianza elegido en val (el de máximo F1) que se aplica tal cual a test: es el punto de operación que tendría el detector desplegado.

In [ ]:
_IMAGE_CACHE = {}


def read_image_tensor(path):
    return decode_image(read_file(str(path)), mode=ImageReadMode.RGB)


def cached_image(img_id):
    if img_id not in _IMAGE_CACHE:
        _IMAGE_CACHE[img_id] = read_image_tensor(image_path(img_id))
    return _IMAGE_CACHE[img_id]


class DetectionDataset(torch.utils.data.Dataset):
    """Imágenes uint8 en RAM y cajas xyxy en píxeles. paths permite evaluar sobre versiones degradadas del test."""

    def __init__(self, ids, transforms=None, paths=None):
        self.ids = list(ids)
        self.transforms = transforms
        # se cargan antes de crear los workers para que todos compartan la misma memoria
        self.images = ([cached_image(i) for i in self.ids] if paths is None
                       else [read_image_tensor(p) for p in paths])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        image = tv_tensors.Image(self.images[idx])
        boxes, labels = GT[img_id]
        target = {
            "boxes": tv_tensors.BoundingBoxes(torch.from_numpy(boxes), format="XYXY", canvas_size=image.shape[-2:]),
            "labels": torch.from_numpy(labels),
            "image_id": img_id,
            "orig_hw": tuple(image.shape[-2:]),
        }
        if self.transforms is not None:
            image, target = self.transforms(image, target)
        return image, target


def list_collate(batch):
    return [b[0] for b in batch], [b[1] for b in batch]


def stack_collate(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]


def train_channel_stats():
    """Media y desviación por canal (escala 0-255) del train completo; se guardan junto a las particiones."""
    path = SPLITS_DIR / "train_channel_stats.json"
    if path.exists():
        stats = read_json(path)
    else:
        total = torch.zeros(3, dtype=torch.float64)
        total_sq = torch.zeros(3, dtype=torch.float64)
        n_pixels = 0
        for img_id in SPLIT_IDS_FULL["train"]:
            x = read_image_tensor(image_path(img_id)).double()
            total += x.sum((1, 2))
            total_sq += (x ** 2).sum((1, 2))
            n_pixels += x.shape[1] * x.shape[2]
        mean = total / n_pixels
        stats = {"mean": mean.tolist(), "std": (total_sq / n_pixels - mean ** 2).sqrt().tolist()}
        write_json(path, stats)
    return stats


CHANNEL_STATS = train_channel_stats()
print("media RGB del train:", np.round(CHANNEL_STATS["mean"], 1), " desviación:", np.round(CHANNEL_STATS["std"], 1))

In [ ]:
PRED_COLUMNS = ["id", "x1", "y1", "x2", "y2", "score", "label"]


def detections_frame(img_id, boxes, scores, labels):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    return pd.DataFrame({"id": np.full(len(boxes), img_id, dtype=np.int64),
                         "x1": boxes[:, 0], "y1": boxes[:, 1], "x2": boxes[:, 2], "y2": boxes[:, 3],
                         "score": np.asarray(scores, dtype=np.float32), "label": np.asarray(labels, dtype=np.int64)})


def concat_detections(frames):
    frames = [f for f in frames if len(f)]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=PRED_COLUMNS)


def pred_path(model_key, split):
    return PRED_DIR / f"{model_key}_{split}.parquet"


def by_image(pred_df):
    return {img_id: g for img_id, g in pred_df.groupby("id")}


def coco_metrics(pred_df, ids):
    """mAP tipo COCO con torchmetrics (backend faster_coco_eval) + AP por clase + curva PR a IoU 0.5."""
    groups = by_image(pred_df)
    preds, targets = [], []
    for img_id in ids:
        g = groups.get(img_id)
        boxes = g[["x1", "y1", "x2", "y2"]].to_numpy(np.float32) if g is not None else np.zeros((0, 4), np.float32)
        preds.append({"boxes": torch.tensor(boxes),
                      "scores": torch.tensor(g["score"].to_numpy(np.float32) if g is not None else np.zeros(0, np.float32)),
                      "labels": torch.tensor(g["label"].to_numpy(np.int64) if g is not None else np.zeros(0, np.int64))})
        gt_boxes, gt_labels = GT[img_id]
        targets.append({"boxes": torch.from_numpy(gt_boxes), "labels": torch.from_numpy(gt_labels)})
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True,
                                  extended_summary=True, backend="faster_coco_eval")
    metric.update(preds, targets)
    res = metric.compute()

    # precision: [IoU, recall(101), clase, área, maxDets]; -1 marca clases sin cajas reales
    precision = res["precision"][0, :, :, 0, -1].numpy()
    per_class = {}
    map_per_class = torch.atleast_1d(res["map_per_class"])
    for k, cls in enumerate(torch.atleast_1d(res["classes"]).tolist()):
        p = precision[:, k]
        ap50 = float(p[p > -1].mean()) if (p > -1).any() else float("nan")
        ap = float(map_per_class[k])
        if ap >= 0:
            per_class[CLASSES[cls]] = {"ap": ap, "ap50": ap50}
    valid = (precision > -1).all(0)
    pr_curve = precision[:, valid].mean(1).tolist() if valid.any() else []
    out = {k: float(res[k]) for k in ("map", "map_50", "map_75", "map_small", "map_medium", "map_large", "mar_100")}
    out.update(per_class=per_class, pr_curve=pr_curve)
    return out


def match_detections(pred_df, ids, iou_thr):
    """Emparejamiento greedy por puntaje como en COCO: una caja real solo puede validar una detección de su clase."""
    groups = by_image(pred_df)
    scores, is_tp = [], []
    n_gt = 0
    for img_id in ids:
        gt_boxes, gt_labels = GT[img_id]
        n_gt += len(gt_labels)
        g = groups.get(img_id)
        if g is None:
            continue
        g = g.sort_values("score", ascending=False)
        tp = np.zeros(len(g), dtype=bool)
        if len(gt_labels):
            ious = box_iou(torch.tensor(g[["x1", "y1", "x2", "y2"]].to_numpy(np.float32)),
                           torch.from_numpy(gt_boxes)).numpy()
            ious[g["label"].to_numpy()[:, None] != gt_labels[None, :]] = 0
            matched = np.zeros(len(gt_labels), dtype=bool)
            for i in range(len(g)):
                candidates = np.where(matched, -1.0, ious[i])
                j = int(candidates.argmax())
                if candidates[j] >= iou_thr:
                    tp[i], matched[j] = True, True
        scores.append(g["score"].to_numpy())
        is_tp.append(tp)
    if not scores:
        return np.zeros(0), np.zeros(0, dtype=bool), n_gt
    return np.concatenate(scores), np.concatenate(is_tp), n_gt


def prf_at(scores, is_tp, n_gt, threshold):
    keep = scores >= threshold
    n_det, n_tp = int(keep.sum()), int(is_tp[keep].sum())
    precision = n_tp / n_det if n_det else 0.0
    recall = n_tp / n_gt if n_gt else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": n_tp, "fp": n_det - n_tp, "fn": n_gt - n_tp}


def best_f1_threshold(scores, is_tp, n_gt):
    grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
    f1 = [prf_at(scores, is_tp, n_gt, t)["f1"] for t in grid]
    return float(grid[int(np.argmax(f1))])


def detection_confusion(pred_df, ids, threshold, iou_thr):
    """Filas = clase real, columnas = clase predicha; el último índice es fondo (detección sin caja real o caja perdida)."""
    groups = by_image(pred_df)
    matrix = np.zeros((NUM_CLASSES + 1, NUM_CLASSES + 1), dtype=np.int64)
    for img_id in ids:
        gt_boxes, gt_labels = GT[img_id]
        g = groups.get(img_id)
        g = g[g["score"] >= threshold] if g is not None else None
        pred_labels = g["label"].to_numpy() if g is not None else np.zeros(0, np.int64)
        used_pred, used_gt = set(), set()
        if len(pred_labels) and len(gt_labels):
            ious = box_iou(torch.tensor(g[["x1", "y1", "x2", "y2"]].to_numpy(np.float32)),
                           torch.from_numpy(gt_boxes)).numpy()
            pairs = np.argwhere(ious >= iou_thr)
            for p, t in pairs[np.argsort(-ious[pairs[:, 0], pairs[:, 1]])]:
                if p not in used_pred and t not in used_gt:
                    used_pred.add(p)
                    used_gt.add(t)
                    matrix[gt_labels[t], pred_labels[p]] += 1
        for t in set(range(len(gt_labels))) - used_gt:
            matrix[gt_labels[t], NUM_CLASSES] += 1
        for p in set(range(len(pred_labels))) - used_pred:
            matrix[NUM_CLASSES, pred_labels[p]] += 1
    return matrix


def evaluate_model(model_key):
    """Métricas de val y test desde las predicciones guardadas; el umbral de operación se fija en val."""
    iou_thr = CONFIG["eval"]["iou_match"]
    val_df, test_df = pd.read_parquet(pred_path(model_key, "val")), pd.read_parquet(pred_path(model_key, "test"))
    val_match = match_detections(val_df, SPLIT_IDS["val"], iou_thr)
    threshold = best_f1_threshold(*val_match)
    test_match = match_detections(test_df, SPLIT_IDS["test"], iou_thr)
    return {
        "threshold": threshold,
        "val": {**coco_metrics(val_df, SPLIT_IDS["val"]), **prf_at(*val_match, threshold)},
        "test": {**coco_metrics(test_df, SPLIT_IDS["test"]), **prf_at(*test_match, threshold)},
    }


def has_predictions(model_key):
    return pred_path(model_key, "val").exists() and pred_path(model_key, "test").exists()

**Verificación del evaluador**: si las predicciones son exactamente las cajas reales (con confianza 1), el mAP debe dar 1.0 y la precisión y el recall también.

In [ ]:
_gt_as_pred = concat_detections([detections_frame(i, GT[i][0], np.ones(len(GT[i][1])), GT[i][1])
                                 for i in SPLIT_IDS["val"]])
_check = coco_metrics(_gt_as_pred, SPLIT_IDS["val"])
_prf = prf_at(*match_detections(_gt_as_pred, SPLIT_IDS["val"], 0.5), 0.5)
print(f"GT como predicción -> mAP50-95 {_check['map']:.3f}, mAP50 {_check['map_50']:.3f}, "
      f"P {_prf['precision']:.3f}, R {_prf['recall']:.3f}")
assert abs(_check["map"] - 1) < 1e-6 and _prf["f1"] == 1.0

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


@torch.no_grad()
def measure_gflops(model, inputs):
    """FLOPs de un forward en inferencia (FlopCounterMode cuenta convoluciones y productos matriciales)."""
    model.eval()
    with FlopCounterMode(display=False) as counter:
        model(inputs)
    return counter.get_total_flops() / 1e9


def measure_latency(predict_one, n_images):
    """ms por imagen de punta a punta (preprocesamiento, forward y postprocesamiento), batch 1, fp32."""
    images = [cached_image(i) for i in SPLIT_IDS["test"][:n_images]]
    for image in images[:min(10, len(images))]:
        predict_one(image)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for image in images:
        predict_one(image)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - t0) / len(images)


def warmup_cosine(total_iters, warmup_iters, min_ratio):
    """Factor de lr por iteración: rampa lineal y luego coseno hasta min_ratio."""
    def factor(it):
        if it < warmup_iters:
            return (it + 1) / warmup_iters
        progress = min(1.0, (it - warmup_iters) / max(1, total_iters - warmup_iters))
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return factor


class DriveMirror:
    """Respaldo en Drive de una carpeta de checkpoints que vive en el disco de la VM."""

    def __init__(self, local_dir, drive_dir):
        self.local_dir, self.drive_dir = Path(local_dir), Path(drive_dir)
        self.last_sync = time.time()

    def restore(self):
        """En una sesión nueva, trae el último respaldo desde Drive para poder reanudar."""
        if not self.local_dir.exists() and self.drive_dir.exists():
            shutil.copytree(self.drive_dir, self.local_dir)
        self.local_dir.mkdir(parents=True, exist_ok=True)

    def sync(self, force=False):
        if not force and time.time() - self.last_sync < 60 * CONFIG["checkpoint_sync_minutes"]:
            return
        for path in self.local_dir.rglob("*"):
            target = self.drive_dir / path.relative_to(self.local_dir)
            if path.is_file() and (not target.exists() or path.stat().st_mtime > target.stat().st_mtime):
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
        self.last_sync = time.time()


def fit(model, loader, compute_loss, optimizer, scheduler, run_dir, epochs, eval_fn,
        eval_every=1, patience=None, grad_clip=None, ema=None):
    """Loop de entrenamiento con AMP, checkpoint reanudable por época y selección por val mAP50-95.

    compute_loss(model, batch) -> dict de pérdidas con la clave "total"; eval_fn(model) -> val mAP50-95.
    Los checkpoints se escriben en CKPT_ROOT y se respaldan en run_dir (Drive); si hay un last.pt continúa desde ahí.
    Al terminar, el mejor modelo (EMA si hay) queda en run_dir/best.pt.
    """
    mirror = DriveMirror(CKPT_ROOT / run_dir.relative_to(RUNS_DIR), run_dir)
    mirror.restore()
    last_path, best_path = mirror.local_dir / "last.pt", mirror.local_dir / "best.pt"
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    state = {"epoch": 0, "best_map": -1.0, "best_epoch": 0, "history": [], "train_time_s": 0.0, "peak_mem_gb": 0.0}
    if last_path.exists():
        ckpt = torch.load(last_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        if ema is not None:
            ema.ema.load_state_dict(ckpt["ema"])
            ema.updates = ckpt["ema_updates"]
        state = ckpt["state"]
        print(f"Reanudando desde la época {state['epoch']} (mejor val mAP50-95 {state['best_map']:.4f})")
    eval_model = ema.ema if ema is not None else model
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(state["epoch"], epochs):
        if patience is not None and state["best_epoch"] and epoch - state["best_epoch"] >= patience:
            print(f"Early stopping: sin mejora desde la época {state['best_epoch']}")
            break
        model.train()
        t0 = time.time()
        sums, n_batches = Counter(), 0
        for batch in loader:
            losses = compute_loss(model, batch)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(losses["total"]).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            if ema is not None:
                ema.update(model)
            for k, v in losses.items():
                sums[k] += v.detach()
            n_batches += 1
        record = {"epoch": epoch + 1, "lr": optimizer.param_groups[0]["lr"],
                  **{f"train_{k}": float(v) / n_batches for k, v in sums.items()}}
        if (epoch + 1) % eval_every == 0 or epoch + 1 == epochs:
            record["val_map"] = float(eval_fn(eval_model))
            if record["val_map"] > state["best_map"]:
                state["best_map"], state["best_epoch"] = record["val_map"], epoch + 1
                atomic_torch_save(eval_model.state_dict(), best_path)
        record["epoch_time_s"] = time.time() - t0
        state["train_time_s"] += record["epoch_time_s"]
        if DEVICE.type == "cuda":
            state["peak_mem_gb"] = max(state["peak_mem_gb"], torch.cuda.max_memory_allocated() / 1e9)
        state["history"].append(record)
        state["epoch"] = epoch + 1
        ckpt = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}
        if ema is not None:
            ckpt.update(ema=ema.ema.state_dict(), ema_updates=ema.updates)
        atomic_torch_save(ckpt, last_path)
        pd.DataFrame(state["history"]).to_csv(run_dir / "history.csv", index=False)
        mirror.sync()
        val_text = f"  val mAP50-95 {record['val_map']:.4f}" if "val_map" in record else ""
        print(f"época {epoch + 1:>3}/{epochs}  loss {record['train_total']:.4f}{val_text}  ({record['epoch_time_s']:.0f} s)")
    mirror.sync(force=True)
    return state


def train_summary(state, extra=None):
    epochs_run = len(state["history"])
    return {"epochs_run": epochs_run, "best_epoch": state["best_epoch"], "best_val_map": state["best_map"],
            "train_time_s": state["train_time_s"], "s_per_epoch": state["train_time_s"] / max(1, epochs_run),
            "peak_gpu_mem_gb": state["peak_mem_gb"], "env": ENV_INFO, **(extra or {})}

In [ ]:
CNN_DIR = RUNS_DIR / "cnn"
YOLO_PROJECT = RUNS_DIR / "yolo"
YOLO_LOCAL = CKPT_ROOT / "yolo"
FRCNN_DIR = RUNS_DIR / "frcnn"
EFFICIENCY_DIR = RUNS_DIR / "efficiency"
for d in (CNN_DIR, YOLO_PROJECT, FRCNN_DIR / "stage_a", FRCNN_DIR / "stage_b", EFFICIENCY_DIR):
    d.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = {
    "cnn": "CNN propia",
    "yolo_scratch": "YOLO11s desde cero",
    "yolo_coco_a": "YOLO11s COCO, etapa A",
    "yolo_coco": "YOLO11s COCO",
    "frcnn_a": "Faster R-CNN COCO, etapa A",
    "frcnn": "Faster R-CNN COCO",
}
FINAL_MODELS = ["cnn", "yolo_scratch", "yolo_coco", "frcnn"]
MODEL_COLORS = {**dict(zip(FINAL_MODELS, PALETTE)), "yolo_coco_a": PALETTE[2], "frcnn_a": PALETTE[3]}
# Directorios de entrenamiento de cada modelo; los de transfer learning acumulan sus dos etapas.
STAGE_DIRS = {
    "cnn": [CNN_DIR],
    "yolo_scratch": [YOLO_PROJECT / "yolo_scratch"],
    "yolo_coco_a": [YOLO_PROJECT / "yolo_coco_a"],
    "yolo_coco": [YOLO_PROJECT / "yolo_coco_a", YOLO_PROJECT / "yolo_coco"],
    "frcnn_a": [FRCNN_DIR / "stage_a"],
    "frcnn": [FRCNN_DIR / "stage_a", FRCNN_DIR / "stage_b"],
}


def is_trained(model_key):
    return (STAGE_DIRS[model_key][-1] / "train_summary.json").exists()


def training_cost(model_key):
    summaries = [read_json(d / "train_summary.json") for d in STAGE_DIRS[model_key] if (d / "train_summary.json").exists()]
    return {"train_time_h": sum(s["train_time_s"] for s in summaries) / 3600,
            "epochs": sum(s["epochs_run"] for s in summaries),
            "best_epoch": summaries[-1]["best_epoch"] if summaries else None,
            "peak_gpu_mem_gb": max((s["peak_gpu_mem_gb"] for s in summaries), default=float("nan"))}


def finalize_model(model_key, predict_ids, efficiency_fn):
    """Predicciones de val/test y medición de eficiencia de un modelo ya entrenado (se omite lo que ya existe)."""
    for split in ("val", "test"):
        if not pred_path(model_key, split).exists():
            predict_ids(SPLIT_IDS[split]).to_parquet(pred_path(model_key, split), index=False)
    eff_path = EFFICIENCY_DIR / f"{model_key}.json"
    if not eff_path.exists():
        write_json(eff_path, efficiency_fn())
    return read_json(eff_path)

## Parte 1: Modelos desde cero

Los dos modelos parten de pesos aleatorios y usan exactamente las mismas particiones (`SPLIT_IDS`), la misma resolución de trabajo (lado largo 640) y el mismo evaluador.

### 1.1 CNN propia: detector anchor-free estilo CenterNet

**Por qué esta arquitectura**: el detector representa cada objeto por su **centro**. La red predice:
- un heatmap por clase, con un pico en el centro de cada pez;
- el ancho y alto de la caja en ese punto;
- un pequeño offset que corrige la discretización.

Frente a un detector con anchors (SSD, YOLOv3) o de dos etapas, esto tiene tres ventajas para este dataset:
1. **No hay anchors que ajustar.** Las cajas van de peces de 10 px a primeros planos de 600 px, y elegir anchors para ese rango sería otro hiperparámetro.
2. **No hace falta NMS.** Un max-pooling 3×3 sobre el heatmap ya deja un solo pico por objeto.
3. **La salida es de alta resolución (stride 4).** Más de la mitad de las cajas son pequeñas según el criterio COCO, y una salida con stride 32 las fundiría en una sola celda.

| Bloque | Diseño | Motivo |
|---|---|---|
| Backbone | stem conv s2 + 4 etapas (64/128/256/384 canales, strides 4-32) con bloques residuales Conv-BN-SiLU | los atajos residuales permiten entrenar desde cero sin que se desvanezca el gradiente; BN estabiliza con lr alto |
| Neck | FPN top-down hasta stride 4, 128 canales | lleva el contexto semántico de stride 32 al mapa de alta resolución donde se ubican los centros |
| Cabezas | 3 ramas Conv3×3-SiLU-Conv1×1: heatmap (20 clases), tamaño (2) y offset (2) | ramas separadas porque clasificar y medir son tareas distintas; el sesgo inicial del heatmap (−2.19 = prior 0.1) evita que la focal loss explote al inicio |
| Pérdida | focal loss con penalización reducida (α=2, β=4) + 0.1·L1 de tamaño + 1·L1 de offset | casi todas las celdas del heatmap son fondo; la focal loss concentra el gradiente en las difíciles y la gaussiana no castiga detecciones a un paso del centro |

El modelo tiene unos 8 M de parámetros, en el mismo orden que YOLO11s (9.4 M). La comparación no queda dominada por el tamaño.

In [ ]:
CNN_STRIDE = 4


def conv_bn_act(c_in, c_out, kernel=3, stride=1):
    return nn.Sequential(nn.Conv2d(c_in, c_out, kernel, stride, kernel // 2, bias=False),
                         nn.BatchNorm2d(c_out), nn.SiLU(inplace=True))


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = conv_bn_act(channels, channels)
        self.conv2 = nn.Sequential(nn.Conv2d(channels, channels, 3, 1, 1, bias=False), nn.BatchNorm2d(channels))
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.conv2(self.conv1(x)))


class CenterNetLite(nn.Module):
    """Backbone residual (strides 4-32) + FPN top-down hasta stride 4 + cabezas de heatmap, tamaño y offset."""

    def __init__(self, num_classes, widths, depths, neck_channels, head_channels):
        super().__init__()
        self.stem = conv_bn_act(3, 32, stride=2)
        stages, c_in = [], 32
        for width, depth in zip(widths, depths):
            stages.append(nn.Sequential(conv_bn_act(c_in, width, stride=2), *[ResidualBlock(width) for _ in range(depth)]))
            c_in = width
        self.stages = nn.ModuleList(stages)
        self.lateral = nn.ModuleList([nn.Conv2d(w, neck_channels, 1) for w in widths])
        self.smooth = nn.ModuleList([conv_bn_act(neck_channels, neck_channels) for _ in widths[:-1]])

        def head(out_channels):
            return nn.Sequential(nn.Conv2d(neck_channels, head_channels, 3, padding=1), nn.SiLU(inplace=True),
                                 nn.Conv2d(head_channels, out_channels, 1))

        self.heatmap_head, self.size_head, self.offset_head = head(num_classes), head(2), head(2)
        nn.init.constant_(self.heatmap_head[-1].bias, -math.log((1 - 0.1) / 0.1))

    def forward(self, x):
        features = []
        x = self.stem(x)
        for stage in self.stages:
            x = stage(x)
            features.append(x)
        p = self.lateral[-1](features[-1])
        for i in range(len(features) - 2, -1, -1):
            p = F.interpolate(p, size=features[i].shape[-2:], mode="nearest") + self.lateral[i](features[i])
            p = self.smooth[i](p)
        return self.heatmap_head(p), self.size_head(p), self.offset_head(p)


def build_cnn():
    cfg = CONFIG["cnn"]
    return CenterNetLite(NUM_CLASSES, cfg["widths"], cfg["depths"], cfg["neck_channels"], cfg["head_channels"])


_cnn_params = count_parameters(build_cnn())[0]
print(f"CenterNetLite: {_cnn_params / 1e6:.2f} M parámetros")

In [ ]:
def gaussian_radius(height, width, min_overlap=0.7):
    """Radio de CornerNet/CenterNet: desplazamiento del centro que todavía deja una caja con IoU >= min_overlap."""
    b1 = height + width
    c1 = width * height * (1 - min_overlap) / (1 + min_overlap)
    r1 = (b1 + math.sqrt(b1 ** 2 - 4 * c1)) / 2
    b2 = 2 * (height + width)
    c2 = (1 - min_overlap) * width * height
    r2 = (b2 + math.sqrt(b2 ** 2 - 16 * c2)) / 2
    a3 = 4 * min_overlap
    b3 = -2 * min_overlap * (height + width)
    c3 = (min_overlap - 1) * width * height
    r3 = (b3 + math.sqrt(b3 ** 2 - 4 * a3 * c3)) / 2
    return min(r1, r2, r3)


def draw_gaussian(heatmap, cx, cy, radius):
    """Dibuja una gaussiana centrada en (cx, cy) quedándose con el máximo donde se superpone con otra."""
    sigma = (2 * radius + 1) / 6
    coords = torch.arange(-radius, radius + 1, dtype=torch.float32)
    gaussian = torch.exp(-(coords[None, :] ** 2 + coords[:, None] ** 2) / (2 * sigma ** 2))
    h, w = heatmap.shape
    left, right = min(cx, radius), min(w - cx, radius + 1)
    top, bottom = min(cy, radius), min(h - cy, radius + 1)
    region = heatmap[cy - top:cy + bottom, cx - left:cx + right]
    torch.maximum(region, gaussian[radius - top:radius + bottom, radius - left:radius + right], out=region)


class CenterNetCollate:
    """Arma el batch y codifica las cajas como heatmaps gaussianos más regresión de tamaño y offset en stride 4."""

    KEYS = ("heatmap", "size", "offset", "index", "mask")

    def __init__(self, num_classes, input_hw, max_objects):
        self.num_classes, self.max_objects = num_classes, max_objects
        self.out_hw = (input_hw[0] // CNN_STRIDE, input_hw[1] // CNN_STRIDE)

    def encode(self, boxes, labels):
        h, w = self.out_hw
        k_max = self.max_objects
        heatmap = torch.zeros(self.num_classes, h, w)
        size, offset = torch.zeros(k_max, 2), torch.zeros(k_max, 2)
        index, mask = torch.zeros(k_max, dtype=torch.long), torch.zeros(k_max)
        for k, (box, cls) in enumerate(zip(boxes[:k_max].tolist(), labels[:k_max].tolist())):
            x1, y1, x2, y2 = (v / CNN_STRIDE for v in box)
            bw, bh = x2 - x1, y2 - y1
            if bw <= 0 or bh <= 0:
                continue
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            ix, iy = min(int(cx), w - 1), min(int(cy), h - 1)
            radius = max(0, int(gaussian_radius(math.ceil(bh), math.ceil(bw))))
            draw_gaussian(heatmap[cls], ix, iy, radius)
            size[k] = torch.tensor([bw, bh])
            offset[k] = torch.tensor([cx - ix, cy - iy])
            index[k] = iy * w + ix
            mask[k] = 1
        return heatmap, size, offset, index, mask

    def __call__(self, batch):
        images = torch.stack([b[0] for b in batch])
        targets = [b[1] for b in batch]
        encoded = [self.encode(t["boxes"], t["labels"]) for t in targets]
        return images, {key: torch.stack([e[i] for e in encoded]) for i, key in enumerate(self.KEYS)}, targets


def focal_loss(logits, target):
    """Focal loss de CenterNet: alpha=2 enfoca en celdas mal clasificadas, (1-y)^4 reduce la penalización cerca del centro."""
    prob = logits.sigmoid()
    pos = target.eq(1).float()
    pos_loss = F.logsigmoid(logits) * (1 - prob) ** 2 * pos
    neg_loss = F.logsigmoid(-logits) * prob ** 2 * (1 - target) ** 4 * (1 - pos)
    return -(pos_loss.sum() + neg_loss.sum()) / pos.sum().clamp(min=1)


def gather_feature(feature, index):
    """feature [B, C, H, W], index [B, K] (posición aplanada) -> [B, K, C]."""
    channels = feature.shape[1]
    return feature.flatten(2).transpose(1, 2).gather(1, index.unsqueeze(-1).expand(-1, -1, channels))


def centernet_loss(outputs, targets, cfg):
    heat, size, offset = (o.float() for o in outputs)
    mask = targets["mask"].unsqueeze(-1).expand(-1, -1, 2)
    n = mask.sum().clamp(min=1)
    heatmap_loss = focal_loss(heat, targets["heatmap"])
    size_loss = (F.l1_loss(gather_feature(size, targets["index"]), targets["size"], reduction="none") * mask).sum() / n
    offset_loss = (F.l1_loss(gather_feature(offset, targets["index"]), targets["offset"], reduction="none") * mask).sum() / n
    total = heatmap_loss + cfg["size_loss_weight"] * size_loss + cfg["offset_loss_weight"] * offset_loss
    return {"total": total, "heatmap": heatmap_loss, "size": size_loss, "offset": offset_loss}


@torch.no_grad()
def centernet_decode(heat_logits, size, offset, k):
    """Top-k picos locales del heatmap (max-pool 3x3 hace de NMS) -> cajas xyxy en píxeles de la entrada."""
    heat = heat_logits.float().sigmoid()
    peaks = heat * (F.max_pool2d(heat, 3, stride=1, padding=1) == heat)
    _, c, h, w = heat.shape
    scores, flat = peaks.flatten(1).topk(min(k, c * h * w))
    labels, pixel = flat // (h * w), flat % (h * w)
    ys, xs = (pixel // w).float(), (pixel % w).float()
    off, wh = gather_feature(offset.float(), pixel), gather_feature(size.float(), pixel).clamp(min=0)
    cx, cy = (xs + off[..., 0]) * CNN_STRIDE, (ys + off[..., 1]) * CNN_STRIDE
    bw, bh = wh[..., 0] * CNN_STRIDE, wh[..., 1] * CNN_STRIDE
    return torch.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], -1), scores, labels

**Aumento de datos y entrada**:
- **Letterbox a 640×384** (múltiplos de 32): la mayoría de los frames son 16:9, así que se desperdicia mucho menos cómputo en relleno que con un cuadrado de 640×640. El relleno va abajo y a la derecha, y para volver a coordenadas de la imagen basta con dividir por la escala.
- **Recorte aleatorio** con el 60-100 % de cada lado: simula peces más cerca de la cámara y entrena la escala.
- **Flip horizontal**: un pez mirando a la izquierda sigue siendo la misma especie. No se usa flip vertical porque el fondo marino y la columna de agua tienen una orientación natural.
- **Jitter de color suave** (±20 % de brillo, contraste y saturación; tono ±0.02): la dominante verde-azul cambia entre sitios y profundidades, pero el color también ayuda a distinguir especies (por ejemplo, los machos de corkwing), así que no se altera mucho.
- **No se usa mosaic** como en YOLO, para mantener el pipeline simple y explicable. Es una diferencia que se discute al comparar.

In [ ]:
class RandomZoomCrop(nn.Module):
    """Recorte aleatorio que conserva entre min_scale y 100 % de cada lado (simula acercamientos)."""

    def __init__(self, min_scale, p):
        super().__init__()
        self.min_scale, self.p = min_scale, p

    def forward(self, image, target):
        if torch.rand(1).item() >= self.p:
            return image, target
        h, w = image.shape[-2:]
        s = self.min_scale + (1 - self.min_scale) * torch.rand(1).item()
        ch, cw = max(1, int(h * s)), max(1, int(w * s))
        top = int(torch.randint(0, h - ch + 1, (1,)))
        left = int(torch.randint(0, w - cw + 1, (1,)))
        return TF.crop(image, top, left, ch, cw), {**target, "boxes": TF.crop(target["boxes"], top, left, ch, cw)}


class Letterbox(nn.Module):
    """Escala sin deformar para caber en (alto, ancho) y rellena abajo/derecha; guarda la escala para invertir."""

    def __init__(self, size_hw, fill=114):
        super().__init__()
        self.size_hw, self.fill = tuple(size_hw), fill

    def forward(self, image, target):
        h, w = image.shape[-2:]
        scale = min(self.size_hw[0] / h, self.size_hw[1] / w)
        nh, nw = min(self.size_hw[0], round(h * scale)), min(self.size_hw[1], round(w * scale))
        padding = [0, 0, self.size_hw[1] - nw, self.size_hw[0] - nh]
        image = TF.pad(TF.resize(image, [nh, nw], antialias=True), padding, fill=self.fill)
        boxes = TF.pad(TF.resize(target["boxes"], [nh, nw]), padding)
        return image, {**target, "boxes": boxes, "scale": (nw / w, nh / h)}


def cnn_train_transforms(cfg):
    return T.Compose([
        RandomZoomCrop(cfg["crop_min_scale"], cfg["crop_prob"]),
        T.RandomHorizontalFlip(0.5),
        T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02)], p=0.5),
        Letterbox(cfg["input_hw"]),
        T.ClampBoundingBoxes(),
        T.SanitizeBoundingBoxes(min_size=2),
    ])


CNN_MEAN = torch.tensor(CHANNEL_STATS["mean"], device=DEVICE).view(1, 3, 1, 1)
CNN_STD = torch.tensor(CHANNEL_STATS["std"], device=DEVICE).view(1, 3, 1, 1)


def normalize_batch(images):
    return (images.to(DEVICE, non_blocking=True).float() - CNN_MEAN) / CNN_STD


def cnn_train_loader():
    cfg = CONFIG["cnn"]
    return DataLoader(DetectionDataset(SPLIT_IDS["train"], cnn_train_transforms(cfg)),
                      batch_size=cfg["batch_size"], shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                      collate_fn=CenterNetCollate(NUM_CLASSES, cfg["input_hw"], cfg["max_objects"]),
                      pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0)


def cnn_compute_loss(model, batch):
    images, targets, _ = batch
    targets = {k: v.to(DEVICE, non_blocking=True) for k, v in targets.items()}
    with torch.autocast(DEVICE.type, enabled=USE_AMP):
        outputs = model(normalize_batch(images))
    return centernet_loss(outputs, targets, CONFIG["cnn"])


@torch.no_grad()
def predict_cnn(model, ids, paths=None, batch_size=16):
    loader = DataLoader(DetectionDataset(ids, Letterbox(CONFIG["cnn"]["input_hw"]), paths), batch_size=batch_size,
                        shuffle=False, num_workers=NUM_WORKERS, collate_fn=stack_collate)
    model.eval()
    frames = []
    for images, targets in loader:
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            outputs = model(normalize_batch(images))
        boxes, scores, labels = (x.cpu() for x in centernet_decode(*outputs, k=CONFIG["eval"]["max_det"]))
        for b, s, l, t in zip(boxes, scores, labels, targets):
            keep = s >= CONFIG["eval"]["conf_min"]
            sx, sy = t["scale"]
            h, w = t["orig_hw"]
            b = b[keep] / torch.tensor([sx, sy, sx, sy])
            b[:, 0::2] = b[:, 0::2].clamp(0, w)
            b[:, 1::2] = b[:, 1::2].clamp(0, h)
            frames.append(detections_frame(t["image_id"], b.numpy(), s[keep].numpy(), l[keep].numpy()))
    return concat_detections(frames)


class ModelEMA:
    """Promedio exponencial de los pesos; el decay crece con las actualizaciones para no arrastrar la inicialización."""

    def __init__(self, model, decay, tau=2000):
        self.ema = copy.deepcopy(model).eval()
        self.decay, self.tau, self.updates = decay, tau, 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = self.decay * (1 - math.exp(-self.updates / self.tau))
        current = model.state_dict()
        for name, value in self.ema.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(d).add_(current[name].detach(), alpha=1 - d)
            else:
                value.copy_(current[name])


def param_groups(model, weight_decay):
    """Sin weight decay en sesgos ni en parámetros de BatchNorm (tensores 1D): penalizarlos no regulariza nada útil."""
    decay = [p for p in model.parameters() if p.requires_grad and p.ndim > 1]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.ndim <= 1]
    return [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]

**Verificación de los targets**: un batch de entrenamiento con sus cajas y el heatmap que la red debe reproducir (máximo sobre las clases). Cada objeto aparece como una gaussiana en stride 4, con un radio proporcional a su tamaño.

In [ ]:
_loader = DataLoader(DetectionDataset(SPLIT_IDS["train"][:64], cnn_train_transforms(CONFIG["cnn"])), batch_size=4,
                     shuffle=True, collate_fn=CenterNetCollate(NUM_CLASSES, CONFIG["cnn"]["input_hw"], CONFIG["cnn"]["max_objects"]))
_images, _targets, _raw = next(iter(_loader))
fig, axes = plt.subplots(2, 4, figsize=(16, 5))
for col in range(4):
    axes[0, col].imshow(_images[col].permute(1, 2, 0).numpy())
    draw_boxes(axes[0, col], _raw[col]["boxes"].tolist(), PALETTE[0], [CLASSES[c] for c in _raw[col]["labels"].tolist()])
    axes[1, col].imshow(_targets["heatmap"][col].max(0).values.numpy(), cmap=SEQ_CMAP, vmin=0, vmax=1)
    for ax in axes[:, col]:
        ax.axis("off")
axes[0, 0].set_title("entrada aumentada (640x384)", loc="left")
axes[1, 0].set_title("heatmap objetivo (160x96)", loc="left")
fig.tight_layout()
save_fig(fig, results_dir("parte1") / "cnn_targets.png")

**Prueba de sobreajuste a un batch**: antes de gastar horas de GPU se verifica que modelo, pérdida y decodificación estén bien conectados. Con 8 imágenes fijas y sin regularización la pérdida debe caer cerca de cero, y el mAP50 sobre esas mismas imágenes debe acercarse a 1.

In [ ]:
def overfit_one_batch(iters):
    cfg = CONFIG["cnn"]
    set_seed(CONFIG["seed"])
    ids = [i for i in SPLIT_IDS["train"] if len(GT[i][1])][:8]
    dataset = DetectionDataset(ids, Letterbox(cfg["input_hw"]))
    batch = CenterNetCollate(NUM_CLASSES, cfg["input_hw"], cfg["max_objects"])([dataset[i] for i in range(len(dataset))])
    model = build_cnn().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.0)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    losses = []
    for _ in range(iters):
        model.train()
        loss = cnn_compute_loss(model, batch)["total"]
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    return {"losses": losses, "batch_map50": coco_metrics(predict_cnn(model, ids), ids)["map_50"]}


overfit_path = CNN_DIR / "overfit_check.json"
if not overfit_path.exists() and CONFIG["run"]["cnn"]:
    write_json(overfit_path, overfit_one_batch(CONFIG["cnn"]["overfit_iters"]))
if overfit_path.exists():
    overfit = read_json(overfit_path)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(overfit["losses"], color=PALETTE[0])
    ax.set_yscale("log")
    ax.set_xlabel("iteración")
    ax.set_title(f"Sobreajuste a 8 imágenes: mAP50 final sobre ellas = {overfit['batch_map50']:.2f}")
    save_fig(fig, results_dir("parte1") / "cnn_overfit_check.png")

**Búsqueda del learning rate (LR range test)**: se entrena unos cientos de iteraciones subiendo el lr exponencialmente desde 1e-6 y se registra la pérdida suavizada. El lr elegido está una década por debajo del punto en que la pérdida empieza a divergir: es la zona donde todavía baja rápido, pero con margen de estabilidad.

In [ ]:
def lr_range_test(iters, lr_min=1e-6, lr_max=1.0):
    cfg = CONFIG["cnn"]
    set_seed(CONFIG["seed"])
    model = build_cnn().to(DEVICE)
    optimizer = torch.optim.AdamW(param_groups(model, cfg["weight_decay"]), lr=lr_min)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    gamma = (lr_max / lr_min) ** (1 / iters)
    records, avg, best, done = [], 0.0, float("inf"), False
    while not done:
        for batch in cnn_train_loader():
            model.train()
            loss = cnn_compute_loss(model, batch)["total"]
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            avg = 0.9 * avg + 0.1 * loss.item()
            smoothed = avg / (1 - 0.9 ** (len(records) + 1))
            records.append({"lr": optimizer.param_groups[0]["lr"], "loss": smoothed})
            best = min(best, smoothed)
            if len(records) >= iters or not math.isfinite(smoothed) or smoothed > 4 * best:
                done = True
                break
            for group in optimizer.param_groups:
                group["lr"] *= gamma
    return pd.DataFrame(records)


lr_test_path = CNN_DIR / "lr_range_test.csv"
if not lr_test_path.exists() and CONFIG["run"]["cnn"]:
    lr_range_test(CONFIG["cnn"]["lr_test_iters"]).to_csv(lr_test_path, index=False)
if lr_test_path.exists():
    lr_test = pd.read_csv(lr_test_path)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(lr_test["lr"], lr_test["loss"], color=PALETTE[0])
    ax.axvline(CONFIG["cnn"]["lr"], color="#52514e", linewidth=1)
    ax.set_xscale("log")
    ax.set_xlabel("learning rate")
    ax.set_ylabel("pérdida suavizada")
    ax.set_title(f"LR range test (línea: lr elegido = {CONFIG['cnn']['lr']:g})")
    save_fig(fig, results_dir("parte1") / "cnn_lr_range_test.png")

**Entrenamiento**:

| Elemento | Valor | Motivo |
|---|---|---|
| Optimizador | AdamW, lr 1e-3 | Adam tolera la escala muy distinta de los gradientes de las tres cabezas; el lr sale del LR range test |
| Weight decay | 0.05, desacoplado y sin aplicarlo a sesgos ni BN | principal regularizador de pesos; con AdamW, 1e-4 casi no tendría efecto |
| Schedule | warmup lineal de 3 épocas y luego coseno hasta 1 % | el warmup evita pasos enormes con las estadísticas de BN aún sin estabilizar; el coseno afina al final |
| EMA de pesos | decay 0.9998 | promedia la trayectoria y suaviza el ruido del batch; YOLO usa lo mismo, así la comparación es justa |
| Épocas | máximo 120, evaluando val cada 5, early stopping con paciencia 30 | desde cero hacen falta muchas épocas; la curva de val decide cuándo parar y se guarda el mejor checkpoint |
| Batch | 16 con AMP (fp16) | cabe holgado en la T4; AMP casi duplica el rendimiento |
| Otros | gradient clipping 10 | protege de picos de la focal loss al inicio |

No se usa dropout: el modelo es convolucional y tiene BN, y el dropout en detectores empeora la localización. La regularización viene del weight decay, el aumento de datos, el EMA y el early stopping.

In [ ]:
def cnn_val_map(model):
    return coco_metrics(predict_cnn(model, SPLIT_IDS["val"]), SPLIT_IDS["val"])["map"]


if CONFIG["run"]["cnn"] and not is_trained("cnn"):
    cfg = CONFIG["cnn"]
    set_seed(CONFIG["seed"])
    model = build_cnn().to(DEVICE)
    ema = ModelEMA(model, cfg["ema_decay"])
    loader = cnn_train_loader()
    optimizer = torch.optim.AdamW(param_groups(model, cfg["weight_decay"]), lr=cfg["lr"])
    scheduler = LambdaLR(optimizer, warmup_cosine(cfg["epochs"] * len(loader), cfg["warmup_epochs"] * len(loader),
                                                  cfg["min_lr_ratio"]))
    state = fit(model, loader, cnn_compute_loss, optimizer, scheduler, CNN_DIR, cfg["epochs"], cnn_val_map,
                eval_every=cfg["eval_every"], patience=cfg["patience"], grad_clip=cfg["grad_clip"], ema=ema)
    write_json(CNN_DIR / "train_summary.json", train_summary(state, {"config": cfg}))

In [ ]:
def load_cnn(weights):
    model = build_cnn().to(DEVICE)
    model.load_state_dict(torch.load(weights, map_location=DEVICE))
    return model.eval()


def cnn_efficiency(model):
    letterbox = Letterbox(CONFIG["cnn"]["input_hw"])

    @torch.no_grad()
    def predict_one(image):
        h, w = image.shape[-2:]
        empty = tv_tensors.BoundingBoxes(torch.zeros(0, 4), format="XYXY", canvas_size=(h, w))
        x, t = letterbox(tv_tensors.Image(image), {"boxes": empty})
        boxes, scores, labels = centernet_decode(*model(normalize_batch(x[None])), k=CONFIG["eval"]["max_det"])
        sx, sy = t["scale"]
        return boxes[0].cpu() / torch.tensor([sx, sy, sx, sy]), scores[0].cpu(), labels[0].cpu()

    return {"params_m": count_parameters(model)[0] / 1e6,
            "gflops_640": measure_gflops(model, torch.zeros(1, 3, 640, 640, device=DEVICE)),
            "latency_ms": measure_latency(predict_one, CONFIG["eval"]["latency_images"])}


if is_trained("cnn"):
    _cnn = load_cnn(CNN_DIR / "best.pt")
    print(json.dumps(finalize_model("cnn", lambda ids: predict_cnn(_cnn, ids), lambda: cnn_efficiency(_cnn)), indent=2))
    print({k: v for k, v in read_json(CNN_DIR / "train_summary.json").items() if k not in ("config", "env")})

### 1.2 YOLO11s desde cero

**Por qué YOLO11s y cómo se entrena**:
- **Arquitectura.** Es la misma que se usa en la Parte 2 con pesos de COCO. Así la diferencia entre ambos experimentos es *solo* la inicialización. Se construye desde `yolo11s.yaml`, sin pesos (la celda siguiente lo verifica comparando la primera convolución con la de COCO). Las piezas son:
  - backbone CSP con bloques C3k2;
  - SPPF para contexto multiescala;
  - C2PSA (atención espacial);
  - neck PAN;
  - cabeza Detect anchor-free y desacoplada (clasificación y regresión con DFL en ramas separadas).
- **Tamaño.** La variante *s* (9.4 M parámetros) es comparable a la CNN propia (unos 8 M). La *n* quedaría corta para 20 clases de grano fino y la *m* duplicaría el costo sin margen en Colab gratis.
- **Optimizador.** SGD (lr0 0.01, momentum 0.937, weight decay 5e-4) es la receta con que Ultralytics entrena desde cero. Se fija explícitamente porque `optimizer="auto"` ignora lr0 y elige otro optimizador según el número de iteraciones.
- **Épocas.** 200, con paciencia 50. Sin pesos previos la convergencia es lenta (COCO desde cero usa 300-600 épocas), y la paciencia alta evita cortar en una meseta temporal.
- **Aumento de datos.** El pipeline estándar de Ultralytics:
  - mosaic, que combina 4 imágenes, multiplica el contexto y la cantidad de objetos chicos por batch;
  - HSV, flip horizontal, escala ±50 % y traslación ±10 %;
  - mosaic se apaga en las últimas 15 épocas para que el modelo se ajuste a imágenes con estadísticas reales;
  - mixup no se usa, porque superpone peces semitransparentes que no existen en el dominio.
- **Regularización.** Weight decay, aumento de datos, EMA de pesos (interno de Ultralytics) y early stopping por fitness de val.

In [ ]:
def first_conv_weights(source):
    return YOLO(source).model.model[0].conv.weight.detach().float()


_w_scratch, _w_coco = first_conv_weights(CONFIG["yolo_scratch"]["model"]), first_conv_weights(CONFIG["yolo_coco"]["model"])
print(f"primera conv desde yaml: std {_w_scratch.std():.4f} | COCO: std {_w_coco.std():.4f} | "
      f"diferencia media absoluta {(_w_scratch - _w_coco).abs().mean():.4f}")

In [ ]:
def read_yolo_results(run_dir):
    df = pd.read_csv(run_dir / "results.csv")
    df.columns = df.columns.str.strip()
    return df


def yolo_train_summary(run_dir):
    df = read_yolo_results(run_dir)
    elapsed = df["time"].to_numpy(float)
    step = np.diff(elapsed, prepend=0.0)
    # el contador de tiempo de Ultralytics se reinicia al reanudar: esa época se toma con su valor completo
    train_time = float(np.where(step > 0, step, elapsed).sum())
    best = df["metrics/mAP50-95(B)"].idxmax()
    peak_path = run_dir / "peak_mem.json"
    return {"epochs_run": len(df), "best_epoch": int(df.loc[best, "epoch"]),
            "best_val_map": float(df.loc[best, "metrics/mAP50-95(B)"]), "train_time_s": train_time,
            "s_per_epoch": train_time / len(df),
            "peak_gpu_mem_gb": read_json(peak_path)["peak_gpu_mem_gb"] if peak_path.exists() else float("nan"),
            "env": ENV_INFO}


def log_peak_memory(trainer):
    if torch.cuda.is_available():
        path = Path(trainer.save_dir) / "peak_mem.json"
        previous = read_json(path)["peak_gpu_mem_gb"] if path.exists() else 0.0
        write_json(path, {"peak_gpu_mem_gb": max(previous, torch.cuda.max_memory_reserved() / 1e9)})


def train_yolo(name, source, stage_cfg):
    """Entrena (o reanuda) una corrida de Ultralytics en el disco de la VM con respaldo periódico en Drive.

    Un last.pt sin optimizador indica que la corrida terminó (Ultralytics lo elimina al cerrar el entrenamiento).
    """
    run_dir = YOLO_PROJECT / name
    if (run_dir / "train_summary.json").exists():
        return run_dir
    mirror = DriveMirror(YOLO_LOCAL / name, run_dir)
    mirror.restore()
    last = mirror.local_dir / "weights" / "last.pt"
    finished = last.exists() and torch.load(last, map_location="cpu", weights_only=False).get("optimizer") is None
    if not finished:
        model = YOLO(str(last) if last.exists() else str(source))
        model.add_callback("on_train_epoch_end", log_peak_memory)
        model.add_callback("on_fit_epoch_end", lambda trainer: mirror.sync())
        if last.exists():
            model.train(resume=True)
        else:
            args = {**CONFIG["yolo_common"], **{k: v for k, v in stage_cfg.items() if k != "model"}}
            model.train(data=str(DATA_YAML), project=str(YOLO_LOCAL), name=name, exist_ok=True, **args)
    mirror.sync(force=True)
    write_json(run_dir / "train_summary.json", {**yolo_train_summary(run_dir), "config": stage_cfg})
    return run_dir


def yolo_weights(name):
    return YOLO_PROJECT / name / "weights" / "best.pt"


def predict_yolo(weights, ids, paths=None):
    model = YOLO(str(weights))
    sources = [str(p) for p in (paths or [image_path(i) for i in ids])]
    results = model.predict(sources, imgsz=CONFIG["yolo_common"]["imgsz"], conf=CONFIG["eval"]["conf_min"],
                            iou=CONFIG["eval"]["nms_iou"], max_det=CONFIG["eval"]["max_det"], batch=16,
                            stream=True, verbose=False)
    return concat_detections([detections_frame(img_id, r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                                               r.boxes.cls.cpu().numpy().astype(np.int64))
                              for img_id, r in zip(ids, results)])


def yolo_efficiency(weights):
    model = YOLO(str(weights))
    # copia sin fusionar Conv+BN para contar parámetros y FLOPs igual que en los otros modelos
    net = copy.deepcopy(model.model).float().to(DEVICE)
    predict_kwargs = dict(imgsz=CONFIG["yolo_common"]["imgsz"], conf=CONFIG["eval"]["conf_min"],
                          iou=CONFIG["eval"]["nms_iou"], max_det=CONFIG["eval"]["max_det"], verbose=False)

    def predict_one(image):
        # Ultralytics interpreta los arreglos numpy como BGR
        return model.predict(np.ascontiguousarray(image.permute(1, 2, 0).numpy()[:, :, ::-1]), **predict_kwargs)

    latency = measure_latency(predict_one, CONFIG["eval"]["latency_images"])
    val = model.val(data=str(DATA_YAML), split="test", batch=16, plots=False, project=str(weights.parent.parent),
                    name="val_test", exist_ok=True, **predict_kwargs)
    return {"params_m": count_parameters(net)[0] / 1e6,
            "gflops_640": measure_gflops(net, torch.zeros(1, 3, 640, 640, device=DEVICE)),
            "latency_ms": latency,
            "ultralytics_test_map": float(val.box.map), "ultralytics_test_map50": float(val.box.map50)}

In [ ]:
if CONFIG["run"]["yolo_scratch"]:
    train_yolo("yolo_scratch", CONFIG["yolo_scratch"]["model"], CONFIG["yolo_scratch"])
if is_trained("yolo_scratch"):
    _w = yolo_weights("yolo_scratch")
    print(json.dumps(finalize_model("yolo_scratch", lambda ids: predict_yolo(_w, ids), lambda: yolo_efficiency(_w)), indent=2))

### 1.3 Comparación de los modelos desde cero

Las funciones de esta celda se reutilizan en la Parte 2. Todas trabajan sobre las predicciones guardadas, de modo que esta sección se puede regenerar sin GPU.

In [ ]:
EVAL = {}


def get_eval(model_key):
    if model_key not in EVAL and has_predictions(model_key):
        EVAL[model_key] = evaluate_model(model_key)
        write_json(results_dir("metricas") / f"{model_key}.json", EVAL[model_key])
    return EVAL.get(model_key)


def efficiency(model_key):
    path = EFFICIENCY_DIR / f"{model_key}.json"
    return read_json(path) if path.exists() else {}


def comparison_table(keys):
    rows = []
    for key in keys:
        ev = get_eval(key)
        if ev is None:
            continue
        m, cost, eff = ev["test"], training_cost(key), efficiency(key)
        rows.append({"modelo": MODEL_NAMES[key], "mAP50-95": m["map"], "mAP50": m["map_50"], "mAP75": m["map_75"],
                     "mAP_pequeños": m["map_small"], "mAP_medianos": m["map_medium"], "mAP_grandes": m["map_large"],
                     "precisión": m["precision"], "recall": m["recall"], "F1": m["f1"], "umbral_conf": ev["threshold"],
                     "val_mAP50-95": ev["val"]["map"], "params_M": eff.get("params_m"), "GFLOPs_640": eff.get("gflops_640"),
                     "latencia_ms": eff.get("latency_ms"), "entrenamiento_h": cost["train_time_h"],
                     "épocas": cost["epochs"], "mejor_época": cost["best_epoch"], "mem_GPU_GB": cost["peak_gpu_mem_gb"]})
    return pd.DataFrame(rows).set_index("modelo").round(4) if rows else pd.DataFrame()


def learning_history(model_key):
    """Época acumulada entre etapas, pérdida de train y val mAP50-95 por época."""
    frames, offset = [], 0
    for d in STAGE_DIRS[model_key]:
        if model_key.startswith("yolo") and (d / "results.csv").exists():
            df = read_yolo_results(d)
            h = pd.DataFrame({"epoch": df["epoch"] + offset,
                              "train_loss": df[["train/box_loss", "train/cls_loss", "train/dfl_loss"]].sum(axis=1),
                              "val_map": df["metrics/mAP50-95(B)"]})
        elif not model_key.startswith("yolo") and (d / "history.csv").exists():
            df = pd.read_csv(d / "history.csv")
            h = pd.DataFrame({"epoch": df["epoch"] + offset, "train_loss": df["train_total"],
                              "val_map": df["val_map"] if "val_map" in df else np.nan})
        else:
            continue
        h["stage"] = d.name
        frames.append(h)
        offset = int(h["epoch"].max())
    return pd.concat(frames, ignore_index=True) if frames else None


def plot_learning_curves(keys, path):
    keys = [k for k in keys if learning_history(k) is not None]
    if not keys:
        return
    fig, axes = plt.subplots(1, len(keys) + 1, figsize=(4.2 * (len(keys) + 1), 3.6))
    for ax, key in zip(axes, keys):
        h = learning_history(key)
        ax.plot(h["epoch"], h["train_loss"], color=MODEL_COLORS[key])
        for boundary in h.groupby("stage", sort=False)["epoch"].max().iloc[:-1]:
            ax.axvline(boundary + 0.5, color="#898781", linewidth=1)
        ax.set_title(f"{MODEL_NAMES[key]}\npérdida de train (escala propia)")
        ax.set_xlabel("época")
    ax = axes[-1]
    for key in keys:
        h = learning_history(key).dropna(subset=["val_map"])
        ax.plot(h["epoch"], h["val_map"], color=MODEL_COLORS[key], label=MODEL_NAMES[key], marker="o", markersize=3)
    ax.set_title("val mAP50-95")
    ax.set_xlabel("época")
    ax.legend()
    fig.tight_layout()
    save_fig(fig, path)


def plot_pr_curves(keys, path):
    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    recall = np.linspace(0, 1, 101)
    for key in keys:
        curve = get_eval(key)["test"]["pr_curve"]
        if curve:
            ax.plot(recall, curve, color=MODEL_COLORS[key],
                    label=f"{MODEL_NAMES[key]} (mAP50 {get_eval(key)['test']['map_50']:.3f})")
    ax.set_xlabel("recall")
    ax.set_ylabel("precisión")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_title("Curva precisión-recall en test (IoU 0.5, promedio sobre clases)")
    ax.legend()
    save_fig(fig, path)


def reliable_classes():
    """Clases con suficientes instancias en test como para que su AP sea interpretable."""
    return [CLASSES[k] for k, n in sorted(TEST_INSTANCES.items(), key=lambda kv: -TRAIN_INSTANCES[kv[0]])
            if n >= CONFIG["eval"]["min_test_instances"]]


def plot_per_class_ap(keys, path):
    classes = reliable_classes()
    unreliable = sorted(set(CLASSES) - set(classes))
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(classes) * len(keys) / 2 + 2))
    height = 0.8 / len(keys)
    y = np.arange(len(classes))
    for i, key in enumerate(keys):
        per_class = get_eval(key)["test"]["per_class"]
        ax.barh(y + (i - (len(keys) - 1) / 2) * height, [per_class.get(c, {}).get("ap", 0.0) for c in classes],
                height=height * 0.9, color=MODEL_COLORS[key], label=MODEL_NAMES[key])
    ax.set_yticks(y, [f"{c} (train {TRAIN_INSTANCES[CLASSES.index(c)]})" for c in classes])
    ax.invert_yaxis()
    ax.set_xlabel("AP50-95 en test")
    ax.set_title(f"AP por clase (clases con >= {CONFIG['eval']['min_test_instances']} cajas en test)")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right")
    save_fig(fig, path)
    if unreliable:
        print("Clases con muy pocas cajas en test (AP poco confiable):",
              {c: TEST_INSTANCES.get(CLASSES.index(c), 0) for c in unreliable})


def plot_ap_vs_instances(keys, path):
    fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 3.6), sharey=True, squeeze=False)
    for ax, key in zip(axes[0], keys):
        per_class = get_eval(key)["test"]["per_class"]
        names = [c for c in per_class if TEST_INSTANCES.get(CLASSES.index(c), 0) >= CONFIG["eval"]["min_test_instances"]]
        x = [TRAIN_INSTANCES[CLASSES.index(c)] for c in names]
        ax.scatter(x, [per_class[c]["ap"] for c in names], color=MODEL_COLORS[key], s=24)
        ax.set_xscale("log")
        ax.set_xlabel("cajas de la clase en train")
        ax.set_title(MODEL_NAMES[key])
    axes[0, 0].set_ylabel("AP50-95 en test")
    fig.suptitle("¿Cuánto depende el AP de la cantidad de ejemplos de entrenamiento?")
    fig.tight_layout()
    save_fig(fig, path)


def plot_confusions(keys, path):
    labels = CLASSES + ["fondo"]
    n_cols = min(2, len(keys))
    n_rows = math.ceil(len(keys) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8.5 * n_cols, 7.5 * n_rows), squeeze=False)
    for ax, key in zip(axes.flat, keys):
        ev = get_eval(key)
        matrix = detection_confusion(pd.read_parquet(pred_path(key, "test")), SPLIT_IDS["test"],
                                     ev["threshold"], CONFIG["eval"]["iou_match"])
        normalized = matrix / np.maximum(matrix.sum(1, keepdims=True), 1)
        ax.imshow(normalized, cmap=SEQ_CMAP, vmin=0, vmax=1)
        ax.set_xticks(range(len(labels)), labels, rotation=90)
        ax.set_yticks(range(len(labels)), labels)
        ax.set_xlabel("predicción")
        ax.set_ylabel("clase real")
        ax.set_title(f"{MODEL_NAMES[key]} (filas normalizadas, conf >= {ev['threshold']:.2f})")
        ax.grid(False)
    for ax in list(axes.flat)[len(keys):]:
        ax.axis("off")
    fig.tight_layout()
    save_fig(fig, path)


QUALITATIVE_IDS = sorted(random.Random(CONFIG["seed"]).sample(
    [i for i in SPLIT_IDS["test"] if len(GT[i][1]) >= 2] or SPLIT_IDS["test"], k=min(4, len(SPLIT_IDS["test"]))))


def plot_qualitative(keys, path, ids=None, paths=None, pred_frames=None):
    """Filas = imágenes de test; columnas = cajas reales y predicciones de cada modelo sobre su umbral de val."""
    ids = ids or QUALITATIVE_IDS
    fig, axes = plt.subplots(len(ids), len(keys) + 1, figsize=(4.2 * (len(keys) + 1), 2.6 * len(ids)), squeeze=False)
    frames = pred_frames or {key: pd.read_parquet(pred_path(key, "test")) for key in keys}
    for r, img_id in enumerate(ids):
        image = Image.open(paths[r] if paths else image_path(img_id))
        boxes, labels = GT[img_id]
        axes[r, 0].imshow(image)
        draw_boxes(axes[r, 0], boxes, "#52514e", [CLASSES[c] for c in labels])
        axes[r, 0].set_title(f"id {img_id}: cajas reales" if r == 0 else f"id {img_id}")
        for c, key in enumerate(keys, start=1):
            d = frames[key]
            d = d[(d["id"] == img_id) & (d["score"] >= get_eval(key)["threshold"])]
            axes[r, c].imshow(image)
            draw_boxes(axes[r, c], d[["x1", "y1", "x2", "y2"]].to_numpy(), MODEL_COLORS[key],
                       [f"{CLASSES[l]} {s:.2f}" for l, s in zip(d["label"], d["score"])])
            if r == 0:
                axes[r, c].set_title(MODEL_NAMES[key])
    for ax in axes.flat:
        ax.axis("off")
    fig.tight_layout()
    save_fig(fig, path)


def report(keys, part):
    """Tabla comparativa + todas las figuras de comparación para los modelos con predicciones disponibles."""
    keys = [k for k in keys if get_eval(k) is not None]
    if not keys:
        print("Todavía no hay modelos entrenados para esta comparación.")
        return pd.DataFrame()
    out = results_dir(part)
    table = comparison_table(keys)
    display(table)
    table.to_csv(out / "comparison_table.csv")
    plot_learning_curves(keys, out / "learning_curves.png")
    plot_pr_curves(keys, out / "pr_curves.png")
    plot_per_class_ap(keys, out / "per_class_ap.png")
    plot_ap_vs_instances(keys, out / "ap_vs_train_instances.png")
    plot_confusions(keys, out / "confusion_matrices.png")
    plot_qualitative(keys, out / "qualitative.png")
    write_json(out / "metrics.json", {k: {"eval": get_eval(k), "cost": training_cost(k), "efficiency": efficiency(k)}
                                      for k in keys})
    return table

In [ ]:
table_parte1 = report(["cnn", "yolo_scratch"], "parte1")

## Parte 2: Transfer learning desde COCO

Los dos modelos parten de pesos entrenados en COCO, donde la clase *fish* existe pero es minoritaria. Se entrenan en dos etapas, y cada una se evalúa por separado:
- **Etapa A (backbone congelado)**: solo se ajusta lo que depende de la tarea nueva (neck y cabezas). Los extractores de bordes, texturas y formas de COCO se usan tal cual. Es barato, y con pocos datos no se arriesga a destruir esas características.
- **Etapa B (fine-tuning)**: se parte del mejor checkpoint de A y se descongela el backbone (todo en YOLO; layer2-4 en Faster R-CNN), con un lr unas 5 veces menor. Así las características se adaptan al dominio submarino (color verde-azul, baja nitidez, turbidez) sin olvidar lo aprendido.

La comparación A vs B mide cuánto aporta adaptar el backbone, más allá de reentrenar las cabezas.

### 2.1 YOLO11s preentrenado en COCO

**Decisiones**:
- **Arquitectura y aumento de datos.** Son los mismos de 1.2, de modo que la diferencia con YOLO desde cero es *solo* la inicialización.
- **Etapa A.** `freeze=11` congela el backbone completo de YOLO11 (capas 0-10: convoluciones C3k2, SPPF y C2PSA). Se usa SGD con lr0 0.01 durante 15 épocas: neck y cabeza aprenden rápido sobre características fijas.
- **Etapa B.** Todo se entrena con lr0 0.002 (5 veces menos), un warmup corto de 1 época y lr de sesgos en warmup de 0.002. El default de 0.1 sacude una cabeza que ya está entrenada. Son 50 épocas con paciencia 20: con buena inicialización la curva de val se estabiliza mucho antes que desde cero.
- **Cabeza Detect.** La rama de regresión de cajas (cv2, con DFL) se transfiere completa. La última convolución de clasificación (cv3, 80 → 20 clases) cambia de forma y se reinicializa; la tabla siguiente lo calcula desde los pesos.
- **Ultralytics y BatchNorm.** Ultralytics vuelve a poner todo el modelo en modo train en cada época. Por eso las BatchNorm de las capas congeladas siguen actualizando sus estadísticas, aunque sus pesos no cambien.

In [ ]:
if CONFIG["run"]["yolo_coco"]:
    train_yolo("yolo_coco_a", CONFIG["yolo_coco"]["model"], CONFIG["yolo_coco"]["stage_a"])
    train_yolo("yolo_coco", yolo_weights("yolo_coco_a"), CONFIG["yolo_coco"]["stage_b"])
for _key, _name in (("yolo_coco_a", "yolo_coco_a"), ("yolo_coco", "yolo_coco")):
    if is_trained(_key):
        _w = yolo_weights(_name)
        print(_key, finalize_model(_key, lambda ids, w=_w: predict_yolo(w, ids), lambda w=_w: yolo_efficiency(w)))

In [ ]:
def yolo_component_table():
    coco = YOLO(CONFIG["yolo_coco"]["model"]).model
    ours = YOLO(str(yolo_weights("yolo_coco_a"))).model if is_trained("yolo_coco_a") else None
    reference = ours if ours is not None else coco
    groups = {"backbone (capas 0-10)": range(0, 11), "neck PAN (capas 11-22)": range(11, 23),
              "cabeza Detect (capa 23)": range(23, 24)}
    freeze = CONFIG["yolo_coco"]["stage_a"]["freeze"]
    rows = []
    for name, layers in groups.items():
        params = sum(p.numel() for i in layers for p in reference.model[i].parameters())
        rows.append({"componente": name, "parámetros_M": params / 1e6,
                     "etapa A": "congelado" if max(layers) < freeze else "entrenado",
                     "etapa B": "fine-tuning"})
    table = pd.DataFrame(rows).set_index("componente")
    if ours is not None:
        coco_sd, our_sd = coco.state_dict(), ours.state_dict()
        reinit = [k for k in our_sd if k in coco_sd and coco_sd[k].shape != our_sd[k].shape]
        print(f"Tensores reinicializados por cambio de forma (80 -> {NUM_CLASSES} clases): "
              f"{sum(our_sd[k].numel() for k in reinit):,} parámetros en {reinit}")
    return table


yolo_components = yolo_component_table()
display(yolo_components.round(3))
yolo_components.to_csv(results_dir("parte2") / "yolo_coco_components.csv")

### 2.2 Faster R-CNN ResNet50-FPN v2 preentrenado en COCO

**Por qué este modelo**: es el detector clásico de **dos etapas**. Primero, una Region Proposal Network propone ~1000 regiones candidatas sobre el FPN. Luego, RoIAlign recorta cada una a 7×7 y una cabeza las clasifica y ajusta su caja. Se usa la versión v2 de torchvision, que alcanza 46.7 de box AP en COCO contra 37.0 de la v1 gracias a una receta de entrenamiento mejorada y a una cabeza de cajas con convoluciones y BatchNorm.

**Decisiones**:
- **Clasificador nuevo.** Se reemplaza `box_predictor` por uno de 21 salidas (20 clases + fondo). Las etiquetas se desplazan en +1 porque el índice 0 está reservado al fondo.
- **Misma resolución.** `min_size = max_size = 640` mantiene las imágenes en su tamaño de trabajo (640×360), igual que los demás modelos. El default de torchvision las ampliaría a 800×1333.
- **Cola de baja confianza.** `box_score_thresh = 0.001`, igual que el resto. El default de 0.05 recortaría la cola de la curva PR y bajaría artificialmente el mAP.
- **BatchNorm congelada.** La v2 trae `BatchNorm2d` normal en el backbone (la v1 usaba `FrozenBatchNorm2d`). Con batch 4, las estadísticas de BN se degradarían, así que en el ResNet50 se reemplazan por `FrozenBatchNorm2d` con las estadísticas de COCO. La BN del FPN y de la cabeza sigue entrenando, porque ve miles de regiones por batch.
- **Etapa A.** Todo el ResNet50 está congelado; se entrenan FPN, RPN, la cabeza de cajas y el clasificador nuevo. Se usa SGD con lr 0.005 (regla lineal de torchvision: 0.02 × 4/16), momentum 0.9, weight decay 1e-4 y 500 iteraciones de warmup, durante 5 épocas.
- **Etapa B.** Se descongelan layer2-layer4; conv1 y layer1 siguen congeladas porque detectan bordes genéricos y son las que más se dañan con pocos datos. El lr es de 0.001 con coseno, durante 12 épocas, y se elige la mejor época por val mAP50-95.
- **Aumento de datos.** Solo flip horizontal y jitter de color suave: el fine-tuning es corto y las características de COCO ya son robustas a escala.

In [ ]:
def freeze_batchnorm(module):
    """Reemplaza cada BatchNorm2d por FrozenBatchNorm2d con las mismas estadísticas y pesos."""
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=child.eps)
            for attr in ("weight", "bias", "running_mean", "running_var"):
                getattr(frozen, attr).copy_(getattr(child, attr).detach())
            setattr(module, name, frozen)
        else:
            freeze_batchnorm(child)


def build_frcnn():
    model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, min_size=640, max_size=640,
                                       box_score_thresh=CONFIG["eval"]["conf_min"],
                                       box_detections_per_img=CONFIG["eval"]["max_det"])
    assert isinstance(model.backbone.body.bn1, nn.BatchNorm2d)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,
                                                      NUM_CLASSES + 1)
    freeze_batchnorm(model.backbone.body)
    return model


def set_frcnn_stage(model, stage):
    for p in model.parameters():
        p.requires_grad = True
    body = model.backbone.body
    frozen = ["conv1", "layer1", "layer2", "layer3", "layer4"] if stage == "A" else ["conv1", "layer1"]
    for name in frozen:
        for p in getattr(body, name).parameters():
            p.requires_grad = False


FRCNN_COMPONENTS = {
    "backbone: conv1 + layer1": lambda m: [m.backbone.body.conv1, m.backbone.body.layer1],
    "backbone: layer2-layer4": lambda m: [m.backbone.body.layer2, m.backbone.body.layer3, m.backbone.body.layer4],
    "FPN": lambda m: [m.backbone.fpn],
    "RPN": lambda m: [m.rpn],
    "cabeza de cajas (RoI)": lambda m: [m.roi_heads.box_head],
    "clasificador nuevo (21 salidas)": lambda m: [m.roi_heads.box_predictor],
}


def frcnn_component_table():
    model = build_frcnn()
    rows = {name: {"parámetros_M": sum(p.numel() for mod in get(model) for p in mod.parameters()) / 1e6}
            for name, get in FRCNN_COMPONENTS.items()}
    for stage in ("A", "B"):
        set_frcnn_stage(model, stage)
        for name, get in FRCNN_COMPONENTS.items():
            params = [p for mod in get(model) for p in mod.parameters()]
            rows[name][f"etapa {stage}"] = "entrenado" if all(p.requires_grad for p in params) else "congelado"
    print("BatchNorm del ResNet50:", type(model.backbone.body.bn1).__name__)
    return pd.DataFrame(rows).T.rename_axis("componente")


frcnn_components = frcnn_component_table()
display(frcnn_components)
frcnn_components.to_csv(results_dir("parte2") / "frcnn_components.csv")

In [ ]:
def frcnn_train_transforms():
    return T.Compose([
        T.RandomHorizontalFlip(0.5),
        T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02)], p=0.5),
        T.SanitizeBoundingBoxes(min_size=2),
    ])


def frcnn_compute_loss(model, batch):
    images, targets = batch
    images = [img.to(DEVICE, non_blocking=True).float() / 255 for img in images]
    targets = [{"boxes": t["boxes"].as_subclass(torch.Tensor).to(DEVICE), "labels": (t["labels"] + 1).to(DEVICE)}
               for t in targets]
    with torch.autocast(DEVICE.type, enabled=USE_AMP):
        losses = model(images, targets)
    losses = {k: v.float() for k, v in losses.items()}
    losses["total"] = sum(losses.values())
    return losses


@torch.no_grad()
def predict_frcnn(model, ids, paths=None, batch_size=8):
    loader = DataLoader(DetectionDataset(ids, None, paths), batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=list_collate)
    model.eval()
    frames = []
    for images, targets in loader:
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            outputs = model([img.to(DEVICE).float() / 255 for img in images])
        for out, t in zip(outputs, targets):
            frames.append(detections_frame(t["image_id"], out["boxes"].float().cpu().numpy(),
                                           out["scores"].float().cpu().numpy(), out["labels"].cpu().numpy() - 1))
    return concat_detections(frames)


def frcnn_val_map(model):
    return coco_metrics(predict_frcnn(model, SPLIT_IDS["val"]), SPLIT_IDS["val"])["map"]


def train_frcnn_stage(stage, init_weights=None):
    cfg = CONFIG["frcnn"]
    stage_cfg = cfg[f"stage_{stage.lower()}"]
    run_dir = FRCNN_DIR / f"stage_{stage.lower()}"
    set_seed(CONFIG["seed"])
    model = build_frcnn().to(DEVICE)
    if init_weights is not None:
        model.load_state_dict(torch.load(init_weights, map_location=DEVICE))
    set_frcnn_stage(model, stage)
    loader = DataLoader(DetectionDataset(SPLIT_IDS["train"], frcnn_train_transforms()), batch_size=cfg["batch_size"],
                        shuffle=True, drop_last=True, num_workers=NUM_WORKERS, collate_fn=list_collate,
                        pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=stage_cfg["lr"],
                                momentum=cfg["momentum"], weight_decay=cfg["weight_decay"])
    total_iters = stage_cfg["epochs"] * len(loader)
    scheduler = LambdaLR(optimizer, warmup_cosine(total_iters, min(cfg["warmup_iters"], total_iters // 2),
                                                  cfg["min_lr_ratio"]))
    state = fit(model, loader, frcnn_compute_loss, optimizer, scheduler, run_dir, stage_cfg["epochs"], frcnn_val_map)
    write_json(run_dir / "train_summary.json", train_summary(state, {"config": {**cfg, "stage": stage}}))


def load_frcnn(weights):
    model = build_frcnn().to(DEVICE)
    model.load_state_dict(torch.load(weights, map_location=DEVICE))
    return model.eval()


def frcnn_efficiency(model):
    @torch.no_grad()
    def predict_one(image):
        return model([image.to(DEVICE).float() / 255])

    return {"params_m": count_parameters(model)[0] / 1e6,
            "gflops_640": measure_gflops(model, [torch.zeros(3, 640, 640, device=DEVICE)]),
            "latency_ms": measure_latency(predict_one, CONFIG["eval"]["latency_images"])}


if CONFIG["run"]["frcnn"]:
    if not is_trained("frcnn_a"):
        train_frcnn_stage("A")
    if not is_trained("frcnn"):
        train_frcnn_stage("B", init_weights=FRCNN_DIR / "stage_a" / "best.pt")
for _key, _stage in (("frcnn_a", "stage_a"), ("frcnn", "stage_b")):
    if is_trained(_key):
        _model = load_frcnn(FRCNN_DIR / _stage / "best.pt")
        print(_key, finalize_model(_key, lambda ids, m=_model: predict_frcnn(m, ids), lambda m=_model: frcnn_efficiency(m)))

### 2.3 Backbone congelado vs fine-tuning

In [ ]:
stage_rows = []
for base, key_a, key_b in (("YOLO11s COCO", "yolo_coco_a", "yolo_coco"), ("Faster R-CNN COCO", "frcnn_a", "frcnn")):
    for stage_name, key in (("A: backbone congelado", key_a), ("B: fine-tuning", key_b)):
        ev = get_eval(key)
        if ev is None:
            continue
        cost = training_cost(key)
        stage_rows.append({"modelo": base, "etapa": stage_name, "test mAP50-95": ev["test"]["map"],
                           "test mAP50": ev["test"]["map_50"], "precisión": ev["test"]["precision"],
                           "recall": ev["test"]["recall"], "F1": ev["test"]["f1"],
                           "épocas acumuladas": cost["epochs"], "horas acumuladas": cost["train_time_h"]})
stage_table = pd.DataFrame(stage_rows)
if len(stage_table):
    stage_table = stage_table.set_index(["modelo", "etapa"]).round(4)
    display(stage_table)
    stage_table.to_csv(results_dir("parte2") / "frozen_vs_finetuning.csv")

### 2.4 Comparación de los cuatro modelos

In [ ]:
table_final = report(FINAL_MODELS, "comparacion")

## Parte 3: Robustez frente a degradaciones de la imagen

Se evalúa el modelo con mejor **val** mAP50-95 sobre versiones modificadas del conjunto de prueba. Se selecciona en val y no en test para no sesgar la elección con el mismo conjunto en que se reporta. Hay 5 tipos de degradación, cada uno con 3 niveles de severidad, elegidos por cómo se degrada realmente una imagen submarina:

| Degradación | Modelo | Niveles | Fenómeno que representa |
|---|---|---|---|
| Baja iluminación | `I·k` | k = 0.5, 0.3, 0.15 | más profundidad, atardecer, cámaras sin luz artificial |
| Desenfoque | gaussiano de σ px | σ = 1, 2.5, 5 | foco fuera de rango, movimiento de la cámara o del pez, dispersión hacia adelante |
| Ruido | gaussiano aditivo de σ (0-255) | σ = 10, 25, 50 | ruido del sensor con ISO alto en poca luz |
| Bajo contraste | `μ + α·(I − μ)` | α = 0.6, 0.4, 0.2 | luz difusa y aguas sin sombras |
| Turbidez | `t·I + (1 − t)·B` | t = 0.7, 0.5, 0.3 | partículas en suspensión: modelo simplificado de formación de imagen submarina (Jaffe-McGlamery), donde la luz retrodispersada B, el color medio del agua en train, se suma a la señal atenuada |

Las degradaciones se aplican sobre las imágenes de trabajo de 640 px, que son exactamente las que ven los modelos. Si se aplicaran a la resolución original, el redimensionado atenuaría el ruido y el desenfoque. Cada imagen usa una semilla fija, así que los conjuntos degradados son reproducibles. Las cajas reales no cambian, porque ninguna degradación mueve los objetos.

Se reportan las mismas métricas de las partes 1 y 2. El umbral de confianza es el elegido en val con imágenes limpias, como ocurriría con un detector ya desplegado que se encuentra con agua turbia.

In [ ]:
DEGRADATIONS = list(CONFIG["robustness"]["levels"])
DEGRADATION_NAMES = {"low_light": "baja iluminación", "blur": "desenfoque", "noise": "ruido gaussiano",
                     "contrast": "bajo contraste", "turbidity": "turbidez"}
DEGRADATION_COLORS = dict(zip(DEGRADATIONS, PALETTE))
DEGRADED_DIR = WORK_ROOT / "degraded" / SCALE_TAG
VEIL_COLOR = np.array(CHANNEL_STATS["mean"], dtype=np.float32)


def degrade(image, kind, level, rng):
    """image uint8 HxWx3 RGB -> versión degradada uint8."""
    x = image.astype(np.float32)
    if kind == "low_light":
        x = x * level
    elif kind == "blur":
        x = cv2.GaussianBlur(x, (0, 0), sigmaX=level)
    elif kind == "noise":
        x = x + rng.normal(0.0, level, x.shape)
    elif kind == "contrast":
        x = x.mean() + level * (x - x.mean())
    elif kind == "turbidity":
        x = level * x + (1 - level) * VEIL_COLOR
    else:
        raise ValueError(f"degradación desconocida: {kind}")
    return np.clip(np.rint(x), 0, 255).astype(np.uint8)


def degraded_image(img_id, kind, level_idx):
    image = np.asarray(Image.open(image_path(img_id)).convert("RGB"))
    if level_idx == 0:
        return image
    rng = np.random.default_rng([CONFIG["seed"], img_id, DEGRADATIONS.index(kind), level_idx])
    return degrade(image, kind, CONFIG["robustness"]["levels"][kind][level_idx - 1], rng)


def degraded_paths(kind, level_idx, ids):
    return [DEGRADED_DIR / f"{kind}_{level_idx}" / f"{i}.png" for i in ids]


def build_degraded_set(kind, level_idx, ids):
    """Guarda la variante en PNG (sin pérdida, para no suavizar el ruido) una sola vez."""
    out_dir = DEGRADED_DIR / f"{kind}_{level_idx}"
    if (out_dir / "done").exists():
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    for img_id, path in zip(ids, degraded_paths(kind, level_idx, ids)):
        Image.fromarray(degraded_image(img_id, kind, level_idx)).save(path)
    (out_dir / "done").touch()


def make_predictor(model_key):
    if model_key.startswith("yolo"):
        weights = yolo_weights(model_key)
        return lambda ids, paths: predict_yolo(weights, ids, paths)
    if model_key == "cnn":
        model = load_cnn(CNN_DIR / "best.pt")
        return lambda ids, paths: predict_cnn(model, ids, paths)
    model = load_frcnn(FRCNN_DIR / ("stage_b" if model_key == "frcnn" else "stage_a") / "best.pt")
    return lambda ids, paths: predict_frcnn(model, ids, paths)


def robustness_metrics(pred_df, threshold):
    ids = SPLIT_IDS["test"]
    coco = coco_metrics(pred_df, ids)
    scores, is_tp, n_gt = match_detections(pred_df, ids, CONFIG["eval"]["iou_match"])
    prf = prf_at(scores, is_tp, n_gt, threshold)
    tp_scores = scores[is_tp & (scores >= threshold)]
    return {"mAP50-95": coco["map"], "mAP50": coco["map_50"], "precisión": prf["precision"], "recall": prf["recall"],
            "F1": prf["f1"], "conf_media_TP": float(tp_scores.mean()) if len(tp_scores) else float("nan"),
            "per_class": coco["per_class"]}


def robustness_table(model_key, level_indices, allow_predict):
    """Una fila por (degradación, nivel); el nivel 0 es el test limpio. Predice solo lo que falta en Drive."""
    ids = SPLIT_IDS["test"]
    threshold = get_eval(model_key)["threshold"]
    clean = robustness_metrics(pd.read_parquet(pred_path(model_key, "test")), threshold)
    predictor, rows = None, []
    for kind in DEGRADATIONS:
        rows.append({"modelo": model_key, "degradación": kind, "nivel": 0, "valor": None, **clean})
        for level_idx in level_indices:
            path = PRED_DIR / "robustness" / f"{model_key}_{kind}_{level_idx}.parquet"
            if not path.exists():
                if not allow_predict:
                    continue
                build_degraded_set(kind, level_idx, ids)
                predictor = predictor or make_predictor(model_key)
                predictor(ids, degraded_paths(kind, level_idx, ids)).to_parquet(path, index=False)
            rows.append({"modelo": model_key, "degradación": kind, "nivel": level_idx,
                         "valor": CONFIG["robustness"]["levels"][kind][level_idx - 1],
                         **robustness_metrics(pd.read_parquet(path), threshold)})
    df = pd.DataFrame(rows)
    df["caída_mAP_%"] = 100 * (1 - df["mAP50-95"] / clean["mAP50-95"]) if clean["mAP50-95"] > 0 else np.nan
    return df


val_ranking = {k: get_eval(k)["val"]["map"] for k in FINAL_MODELS if get_eval(k) is not None}
BEST_MODEL = max(val_ranking, key=val_ranking.get) if val_ranking else None
print("val mAP50-95:", {MODEL_NAMES[k]: round(v, 4) for k, v in val_ranking.items()})
print("Modelo evaluado en robustez:", MODEL_NAMES.get(BEST_MODEL))

In [ ]:
_example_id = QUALITATIVE_IDS[0]
fig, axes = plt.subplots(len(DEGRADATIONS), 4, figsize=(14, 2.2 * len(DEGRADATIONS)))
for r, kind in enumerate(DEGRADATIONS):
    for level_idx in range(4):
        ax = axes[r, level_idx]
        ax.imshow(degraded_image(_example_id, kind, level_idx))
        value = "" if level_idx == 0 else f" = {CONFIG['robustness']['levels'][kind][level_idx - 1]}"
        ax.set_title(f"{DEGRADATION_NAMES[kind]}: {'limpia' if level_idx == 0 else f'nivel {level_idx}{value}'}",
                     fontsize=8)
        ax.axis("off")
fig.tight_layout()
save_fig(fig, results_dir("parte3") / "degradation_examples.png")

In [ ]:
ROBUSTNESS = pd.DataFrame()
if BEST_MODEL is not None:
    ROBUSTNESS = robustness_table(BEST_MODEL, [1, 2, 3], allow_predict=CONFIG["run"]["robustness"])
    summary_cols = ["degradación", "nivel", "valor", "mAP50-95", "mAP50", "precisión", "recall", "F1",
                    "conf_media_TP", "caída_mAP_%"]
    display(ROBUSTNESS[summary_cols].round(4))
    ROBUSTNESS.drop(columns="per_class").to_csv(results_dir("parte3") / "robustness_best_model.csv", index=False)
    drop_pivot = ROBUSTNESS[ROBUSTNESS["nivel"] > 0].pivot(index="degradación", columns="nivel", values="caída_mAP_%")
    display(drop_pivot.round(1).rename(index=DEGRADATION_NAMES))

In [ ]:
if len(ROBUSTNESS):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    titles = {"mAP50-95": "mAP50-95", "recall": "recall (umbral fijo de val)",
              "conf_media_TP": "confianza media de los aciertos"}
    for ax, metric in zip(axes, titles):
        for kind in DEGRADATIONS:
            d = ROBUSTNESS[ROBUSTNESS["degradación"] == kind].sort_values("nivel")
            ax.plot(d["nivel"], d[metric], marker="o", markersize=4, color=DEGRADATION_COLORS[kind],
                    label=DEGRADATION_NAMES[kind])
        ax.set_xticks(range(4), ["limpia", "1", "2", "3"])
        ax.set_xlabel("severidad")
        ax.set_title(titles[metric])
    axes[0].legend()
    fig.suptitle(f"{MODEL_NAMES[BEST_MODEL]}: desempeño en test según severidad de la degradación")
    fig.tight_layout()
    save_fig(fig, results_dir("parte3") / "robustness_curves.png")

In [ ]:
if len(ROBUSTNESS) and (ROBUSTNESS["nivel"] == 3).any():
    classes = reliable_classes()
    clean_pc = ROBUSTNESS[ROBUSTNESS["nivel"] == 0].iloc[0]["per_class"]
    change = np.full((len(classes), len(DEGRADATIONS)), np.nan)
    for j, kind in enumerate(DEGRADATIONS):
        severe = ROBUSTNESS[(ROBUSTNESS["degradación"] == kind) & (ROBUSTNESS["nivel"] == 3)]
        if len(severe):
            pc = severe.iloc[0]["per_class"]
            for i, c in enumerate(classes):
                if clean_pc.get(c, {}).get("ap", 0) > 0:
                    change[i, j] = 100 * (pc.get(c, {}).get("ap", 0.0) / clean_pc[c]["ap"] - 1)
    limit = max(np.nanmax(np.abs(change)) if np.isfinite(change).any() else 1.0, 1e-6)
    fig, ax = plt.subplots(figsize=(7.5, 0.4 * len(classes) + 1.8))
    ax.imshow(change, cmap=DIV_CMAP.reversed(), norm=TwoSlopeNorm(vcenter=0, vmin=-limit, vmax=limit), aspect="auto")
    for i in range(len(classes)):
        for j in range(len(DEGRADATIONS)):
            if np.isfinite(change[i, j]):
                ax.text(j, i, f"{change[i, j]:.0f}", ha="center", va="center", fontsize=7, color="#0b0b0b")
    ax.set_xticks(range(len(DEGRADATIONS)), [DEGRADATION_NAMES[k] for k in DEGRADATIONS], rotation=20)
    ax.set_yticks(range(len(classes)), classes)
    ax.set_title("Cambio % del AP50-95 por clase en el nivel más severo (rojo = cae)")
    ax.grid(False)
    save_fig(fig, results_dir("parte3") / "robustness_per_class.png")

In [ ]:
if len(ROBUSTNESS) and (ROBUSTNESS["nivel"] == 3).any():
    ids = QUALITATIVE_IDS
    threshold = get_eval(BEST_MODEL)["threshold"]
    columns = [("limpia", None)] + [(DEGRADATION_NAMES[k], k) for k in DEGRADATIONS]
    fig, axes = plt.subplots(len(ids), len(columns), figsize=(3.3 * len(columns), 2.1 * len(ids)), squeeze=False)
    for c, (title, kind) in enumerate(columns):
        path = pred_path(BEST_MODEL, "test") if kind is None else PRED_DIR / "robustness" / f"{BEST_MODEL}_{kind}_3.parquet"
        preds = pd.read_parquet(path) if path.exists() else pd.DataFrame(columns=PRED_COLUMNS)
        for r, img_id in enumerate(ids):
            d = preds[(preds["id"] == img_id) & (preds["score"] >= threshold)]
            ax = axes[r, c]
            ax.imshow(degraded_image(img_id, kind, 3) if kind else degraded_image(img_id, DEGRADATIONS[0], 0))
            draw_boxes(ax, d[["x1", "y1", "x2", "y2"]].to_numpy(), MODEL_COLORS[BEST_MODEL],
                       [f"{CLASSES[l]} {s:.2f}" for l, s in zip(d["label"], d["score"])])
            if r == 0:
                ax.set_title(f"{title} (nivel 3)" if kind else title, fontsize=9)
            ax.axis("off")
    fig.tight_layout()
    save_fig(fig, results_dir("parte3") / "robustness_qualitative.png")

**Comparación adicional**: los cuatro modelos sobre el nivel más severo de cada degradación. Sirve para ver si el ranking en condiciones limpias se mantiene, y si el preentrenamiento en COCO también entrega robustez.

In [ ]:
robust_all = [robustness_table(k, [3], allow_predict=CONFIG["run"]["robustness"])
              for k in FINAL_MODELS if get_eval(k) is not None]
if robust_all:
    robust_all = pd.concat(robust_all, ignore_index=True).drop_duplicates(["modelo", "degradación", "nivel"])
    robust_all.drop(columns="per_class").to_csv(results_dir("parte3") / "robustness_all_models_severe.csv", index=False)
    severe = robust_all[robust_all["nivel"] == 3]
    clean = robust_all[robust_all["nivel"] == 0].drop_duplicates("modelo").assign(degradación="limpia")
    plot_df = pd.concat([clean, severe]).pivot(index="degradación", columns="modelo", values="mAP50-95")
    plot_df = plot_df.reindex(["limpia"] + DEGRADATIONS).dropna(how="all")
    keys = [k for k in FINAL_MODELS if k in plot_df.columns]
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(plot_df))
    width = 0.8 / len(keys)
    for i, key in enumerate(keys):
        ax.bar(x + (i - (len(keys) - 1) / 2) * width, plot_df[key], width=width * 0.9, color=MODEL_COLORS[key],
               label=MODEL_NAMES[key])
    ax.set_xticks(x, ["limpia"] + [DEGRADATION_NAMES[k] for k in plot_df.index[1:]])
    ax.set_ylabel("mAP50-95 en test")
    ax.set_title("Los cuatro modelos en el nivel más severo de cada degradación")
    ax.grid(axis="x", visible=False)
    ax.legend(ncols=2)
    save_fig(fig, results_dir("parte3") / "robustness_all_models.png")
    display(plot_df[keys].rename(columns=MODEL_NAMES, index=DEGRADATION_NAMES).round(4))

## Cierre

### Costo computacional

Horas de GPU efectivas de cada entrenamiento (incluyen la validación por época), acumulando las dos etapas en los modelos de transfer learning. Todos se entrenaron en la misma GPU (ver `ENV_INFO`), así que los tiempos se pueden comparar directamente.

In [ ]:
cost_rows = []
for key in MODEL_NAMES:
    if not is_trained(key):
        continue
    cost, eff = training_cost(key), efficiency(key)
    cost_rows.append({"modelo": MODEL_NAMES[key], "épocas": cost["epochs"], "mejor_época": cost["best_epoch"],
                      "horas_GPU": cost["train_time_h"],
                      "s_por_época": 3600 * cost["train_time_h"] / max(1, cost["epochs"]),
                      "mem_GPU_GB": cost["peak_gpu_mem_gb"], "latencia_ms": eff.get("latency_ms"),
                      "params_M": eff.get("params_m"), "GFLOPs_640": eff.get("gflops_640")})
cost_table = pd.DataFrame(cost_rows)
if len(cost_table):
    cost_table = cost_table.set_index("modelo").round(3)
    display(cost_table)
    cost_table.to_csv(results_dir("comparacion") / "computational_cost.csv")
    finals = [k for k in FINAL_MODELS if is_trained(k)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    names = [MODEL_NAMES[k] for k in finals]
    colors = [MODEL_COLORS[k] for k in finals]
    axes[0].barh(names, [training_cost(k)["train_time_h"] for k in finals], color=colors)
    axes[0].set_xlabel("horas de GPU")
    axes[0].set_title("Tiempo de entrenamiento")
    tables = {k: get_eval(k) for k in finals}
    axes[1].scatter([efficiency(k).get("latency_ms", np.nan) for k in finals],
                    [tables[k]["test"]["map"] if tables[k] else np.nan for k in finals], color=colors, s=60)
    for k in finals:
        if tables[k]:
            axes[1].annotate(MODEL_NAMES[k], (efficiency(k).get("latency_ms", np.nan), tables[k]["test"]["map"]),
                             textcoords="offset points", xytext=(6, 4), fontsize=8, color="#52514e")
    axes[1].set_xlabel("latencia por imagen (ms, batch 1)")
    axes[1].set_ylabel("test mAP50-95")
    axes[1].set_title("Precisión vs velocidad")
    axes[0].grid(axis="y", visible=False)
    fig.tight_layout()
    save_fig(fig, results_dir("comparacion") / "computational_cost.png")

### Exportación de entregables

Las particiones y las figuras y métricas finales se copian a `entrega/` en Drive (en Colab) o a la carpeta `Tarea_2/` del repositorio (en local). Desde ahí se versionan junto al notebook. Los pesos y las predicciones quedan en Drive.

In [ ]:
if not CONFIG["fast_dev_run"]:
    export_dir = DRIVE_ROOT / "entrega" if IN_COLAB else Path.cwd().parent
    shutil.copytree(SPLITS_DIR, export_dir / "splits", dirs_exist_ok=True)
    shutil.copytree(RESULTS_DIR, export_dir / "results", dirs_exist_ok=True)
    print("Particiones y resultados copiados a", export_dir)